# RAG(Retrieval Augmented Generation)
- [RAG](https://python.langchain.com/v0.1/docs/modules/data_connection/)은 *Retrieval Augmented Generation*의 약자로, **검색 기반 생성 기법**을 의미한다. 이 기법은 LLM이 특정 문서에 기반하여 보다 정확하고 신뢰할 수 있는 답변을 생성할 수 있도록 돕는다.     
- 사용자의 질문에 대해 자체적으로 구축한 데이터베이스(DB)나 외부 데이터베이스에서 질문과 관련된 문서를 검색하고, 이를 질문과 함께 LLM에 전달한다.
- LLM은 같이 전달된 문서를 바탕으로 질문에 대한 답변을 생성한다. 
- 이를 통해 LLM이 학습하지 않은 내용도 다룰 수 있으며, 잘못된 정보를 생성하는 환각 현상(*hallucination*)을 줄일 수 있다.

## RAG와 파인튜닝(Fine Tuning) 비교

### 파인튜닝(Fine Tuning)

- **정의**: 사전 학습(pre-trained)된 LLM에 특정 도메인의 데이터를 추가로 학습시켜 해당 도메인에 특화된 맞춤형 모델로 만드는 방식이다.
- **장점**
  - 특정 도메인에 최적화되어 높은 정확도와 성능을 낼 수 있다.
- **단점**
  - 모델 재학습에 많은 시간과 자원이 필요하다.
  - 새로운 정보가 반영되지 않으며, 이를 위해서는 다시 학습해야 한다.

### RAG

- **정의**: 모델을 다시 학습시키지 않고, 외부 지식 기반에서 정보를 검색하여 실시간으로 답변에 활용하는 방식이다.
- **장점**
  - 최신 정보를 쉽게 반영할 수 있다.
  - 모델을 수정하지 않아도 되므로 효율적이다.
- **단점**
  - 검색된 문서의 품질에 따라 답변의 정확성이 달라질 수 있다.
  - 검색 시스템 구축이 필요하다.

## 정리

| 항목       | 파인튜닝 | RAG |
| -------- | ---- | --- |
| 도메인 최적화  | 가능   | 제한적 |
| 최신 정보 반영 | 불가능  | 가능  |
| 구현 난이도   | 높음   | 보통  |
| 유연성      | 낮음   | 높음  |

- LLM은 학습 당시의 데이터만을 기반으로 작동하므로 최신 정보나 기업 내부 자료와 같은 특정한 지식 기반에 접근할 수 없다.
- 파인튜닝은 시간과 비용이 많이 들고 유지보수가 어렵다.
-	반면, RAG는 기존 LLM을 변경하지 않고도 외부 문서를 통해 그 한계를 보완할 수 있다.
- RAG는 특히 빠르게 변화하는 정보를 다루는 분야(예: 기술 지원, 뉴스, 법률 등)에서 유용하게 활용된다. 반면, 정적인 정보에 대해 높은 정확도가 필요한 경우에는 파인튜닝이 효과적이다.


## RAG 작동 단계
- 크게 "**정보 저장(인덱싱)**", "**검색**, **생성**"의 단계로 나눌 수 있다.
  
### 1. 정보 저장(인덱싱)
RAG는 사전에 정보를 가공하여 **벡터 데이터베이스**(Vector 저장소)에 저장해 두고, 나중에 검색할 수 있도록 준비한다. 이 단계는 다음과 같은 과정으로 이루어진다.

1. **Load (불러오기)**
   - 답변시 참조할 사전 정보를 가진 데이터들을 불러온다.
2. **Split/Chunking (문서 분할)**
   - 긴 텍스트를 일정한 길이의 작은 덩어리(*chunk*)로 나눈다.
   - 이렇게 해야 검색과 생성의 정확도를 높일 수 있다.
3. **Embedding (임베딩)**
   - 각 텍스트 조각을 **임베딩 벡터**로 변환한다.
   - 임베딩 벡터는 그 문서의 의미를 벡터화 한 것으로 질문과 유사한 문서를 찾을 때 인덱스로 사용된다.
4. **Store (저장)**
   - 임베딩된 벡터를 **벡터 데이터베이스**(벡터 저장소)에 저장한다.
   - 벡터 데이터베이스는 유사한 질문이나 문장을 빠르게 찾을 수 있도록 특화된 데이터 저장소이다.
   
![rag](figures/rag1.png)

### 2. 검색, 생성

사용자가 질문을 하면 다음과 같은 절차로 답변이 생성된다.
1. **Retrieve (검색)**
   - 사용자의 질문을 임베딩한 후, 이 질문 벡터와 유사한 context 벡터를 벡터 데이터베이스에서 검색하여 찾는다.
2. **Query (질의 생성)**
   - 벡터 데이터베이스에서 검색된 문서 조각과 사용자의 질문을 함께 **프롬프트**(prompt)로 구성하여 LLM에 전달한다.
3. **Generation (응답 생성)**
   - LLM은 받은 프롬프트에 대한 응답을 생성한다.
   
- **RAG 흐름**
  
![Retrieve and Generation](figures/rag2.png)


# Document Loader
- LLM에게 질의할 때 같이 제공할 Data들을 저장하기 위해 먼저 읽어들인다.(Load)
- 데이터 Resouce는 다양하다.
    - 데이터를 로드(load)하는 방식은 저장된 위치와 형식에 따라 다양하다. 
      - 로컬 컴퓨터(Local Computer)에 저장된 문서
        - 예: CSV, Excel, JSON, TXT 파일 등
      - 데이터베이스(Database)에 저장된 데이터셋
      - 인터넷에 존재하는 데이터
        - 예: 웹에 공개된 API, 웹 페이지에 있는 데이터, 클라우드 스토리지에 저장된 파일 등

![rag_load](figures/rag_load.png)

- 다양한 문서 형식(format)에 맞춰 읽어오는 다양한 **document loader** 들을 Langchain에서 지원한다.
    - 다양한 Resource들로 부터 데이터를 읽기 위해서는 다양한 라이브러리를 이용해 서로 다른 방법으로 읽어야 한다.
    - Langchain은 데이터를 읽는 다양한 방식의 코드를 하나의 interface로 사용 할 수 있도록 지원한다.
    - 다양한 3rd party library(ppt, github 등등 다양한 3rd party lib도 있음. )들과 연동해 다양한 Resource로 부터 데이터를 Loading 할 수 있다.
        - https://docs.langchain.com/oss/python/integrations/document_loaders
- **모든 document loader는 기본적으로 동일한 interface(사용법)로 호출할 수있다.**
- **반환타입**
    - **list[Document]**
    - Load 한 문서는 Document객체에 정보들을 넣는다. 여러 문서를 읽을 수 있기 대문에 list에 묶어서 반환한다.
        - **Document 속성**
            - page_content: 문서의 내용
            - metadata(option): 문서에 대한 메타데이터(정보)를 dict 형태로 저장한다. 
            - id(option): 문서의 고유 id
     
- **주의**
    - Langchain을 이용해 RAG를 구현할 때 **꼭 Langchain의 DocumentLoader를 사용해야 하는 것은 아니다.**
    - DocumentLoader는 데이터를 읽어오는 것을 도와주는 라이브러리일 뿐이다. 다른 라이브러리를 이용해서 읽어 들여도 상관없다. 

In [1]:
!uv pip install langchain-community -U

Resolved 47 packages in 1.23s
 Downloaded numpy
Prepared 5 packages in 1.41s
Uninstalled 5 packages in 244ms
Installed 5 packages in 587ms
 - anyio==4.14.0
 + anyio==4.14.1
 - langsmith==0.8.17
 + langsmith==0.9.1
 - numpy==2.4.6
 + numpy==2.5.0
 - websockets==15.0.1
 + websockets==16.0
 - xxhash==3.7.0
 + xxhash==3.7.1


## 주요 Document Loader

### Text file
- TextLoader 이용

In [10]:
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

# with open(path, "rt", encoding="utf-8") as f:
#     print(f.read()[:100])

# TextLoader를 이용해서 조회
## 객체 생성 - 읽을 파일의 경로를 지정. 
loader = TextLoader(path, encoding="utf-8")

# 읽어오기
docs = loader.load() # 실행시 바로 읽는다. => 반환: List[Document]
# docs = loader.lazy_load() # 읽은 문서를 사용할 때 읽는다. 반환: generator [Document]

print(type(docs), len(docs))
print(type(docs[0]))

# for doc in docs: # lazy_load() -> generator
#     print(type(doc))

<class 'list'> 1
<class 'langchain_core.documents.base.Document'>


In [ ]:
doc = docs[0]
print("문서정보: doc.metadata")
print(doc.metadata)
# 필요한 정보들을 추가할 수있다. LLM에 전달할 프롬프트에 추가할 것들, 검색할 때 사용할 정보들.
# 메타데이터의 키 -> 사전에 설계가 필요.
doc.metadata['category'] = "sports"
doc.metadata['tag'] = ["올림픽", "IOC", "동계올림픽", "하계올림픽"]

print(doc.metadata)

문서정보: doc.metadata
{'source': 'data/olympic.txt'}
{'source': 'data/olympic.txt', 'category': 'sports', 'tag': ['올림픽', 'IOC', '동계올림픽', '하계올림픽']}


In [15]:
print("문서내용: doc.page_content")
print(doc.page_content[:200])

문서내용: doc.page_content
올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열


In [16]:
print("문서 식별자-id: doc.id")
print(doc.id)

문서 식별자-id: doc.id
None


### PDF
- PyPDF, Pymupdf 등 다양한 PDF 문서를 읽어들이는 파이썬의  3rd party library들을 이용해 pdf 문서를 Load 한다.
    - https://python.langchain.com/docs/integrations/document_loaders/#pdfs
- 각 PDF Loader 특징
    -  PyMuPDFLoader
        -   텍스트 뿐 아니라 이미지, 주석등의 정보를 추출하는데 성능이 좋다.
        -   PyMuPDF 라이브러리 기반
    - PyPDFLoader
        - 텍스트를 빠르게 추출 할 수있다.
        - PyPDF2 라이브러리 기반. 경량 라이브러리로 빠르고 큰 파일도 효율적으로 처리한다.
    - PDFPlumberLoader
        - 표와 같은 복잡한 구조의 데이터 처리하는데 강력한 성능을 보여준다. 텍스트, 이미지, 표 등을 모두 추출할 수 있다. 
        - PDFPlumber 라이브러리 기반
- 설치 패키지
    - DocumentLoader와 연동하는 라이브러리들을 설치 해야 한다.
    - `pip install pypdf -qU`
    - `pip install pymupdf -qU`
    - `pip install pdfplumber -qU`

In [17]:
!uv pip install pypdf pymupdf pdfplumber

Resolved 10 packages in 229ms
 Downloaded pypdfium2
 Downloaded pymupdf
 Downloaded pdfminer-six
Prepared 5 packages in 691ms
Installed 5 packages in 146ms
 + pdfminer-six==20260107
 + pdfplumber==0.11.10
 + pymupdf==1.27.2.3
 + pypdf==6.14.2
 + pypdfium2==5.10.1


In [36]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader

path = "data/novel/금_따는_콩밭_김유정.pdf"

# Document Loader 생성
## PyPDF
# loader = PyPDFLoader(path, mode="single") # mode: single(전체를 하나의 문서로 읽기)
# loader = PyPDFLoader(path, mode="page") # mode: page(Page당 하나의 문서로 읽기)

## PyMuPDF
# loader = PyMuPDFLoader(path)

## PDFPlumber
loader = PDFPlumberLoader(path)

# 읽기
docs = loader.load() # list[Document]
print(len(docs))

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

23


In [ ]:
from pprint import pprint
doc = docs[0]
pprint(doc.metadata)
# doc.metadata['author'] = '김유정'

{'Author': 'Unknown',
 'CreationDate': "D:20241124070535+00'00'",
 'Creator': 'Wikisource',
 'ModDate': "D:20241124070537+00'00'",
 'Producer': 'Wikisource',
 'Title': '금 따는 콩밭',
 'file_path': 'data/novel/금_따는_콩밭_김유정.pdf',
 'page': 0,
 'source': 'data/novel/금_따는_콩밭_김유정.pdf',
 'total_pages': 23}


In [38]:
print(doc.page_content)

금 따는 콩밭
Exported from Wikisource on 2024년 11월 24일
1



### Web 문서 로드

#### WebBaseLoader를 이용해 Web 문서로딩

requests와 BeautifulSoup을 이용해 web 페이지의 내용을 크롤링해서 Document로 loading한다.

- 주요 파라미터
  - **web_paths***: str | list[str]
    - 크롤링할 대상 URL
  - **requests_kwargs**: dict
    - requests.get() 에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    - headers, cookies, verify 등 설정 전달
  - **header_template**: dict
    - HTTP Header 에 넣을 값을 dict 로 전달.
  - **encoding**
    - requests의 응답 encoding을 설정 (bs_kwargs의 from_encoding 보다 상위에서 적용됨)
  - **bs_kwargs**
    - BeautifulSoup initializer에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    -  주요 옵션
       - **parse_only**: 요청 페이지에서 특정 요소만 선택해서 가져오기. **SoupStrainer를 사용**한다.
         - BeautifulSoup의 `SoupStrainer` 를 이용해 페이지의 일부분만 가져오기
           - 웹 페이지를 파싱(parse, 구조 분석)할 때, 페이지 전체가 아닌 특정 부분만 필요한 경우가 많다. BeautifulSoup 라이브러리의 SoupStrainer를 사용하면, 원하는 태그나 속성이 있는 요소만 골라서 파싱할 수 있다.
           - BeautifulSoup("html문서", parse_only=Strainer객체)
               - Strainer객체에 지정된 영역에서만 내용 찾는다.
           - `SoupStrainer("태그명")`, `SoupStrainer(["태그명", "태그명"])`
             - 지정한 태그 만 조회
           - `SoupStrainer(name="태그명", attrs={속성명:속성값})`
             -  지정한 태그 중 속성명=속성값인 것만 조회
        - **from_encoding**: Encoding 설정 
          - "from_encoding":"utf-8"
   - **bs_get_text_kwargs**:
     - BeautifulSoup객체.get_text() 에 전달할 파라미터 dict로 전달. (key: parameter변수명, value: 전달할 값)
     - **RAG 구축시 `separator` 와 `strip=True` 으로 설정하는 것이 좋다.** (RAG 품질을 위해 강력히 권장되는 설정이다.)
       -  get_text() 는 기본적으로 태그를 제거하고 텍스트만 이어 붙여 반환한다. `separator=구분자문자` 를 지정하여 추출된 텍스트 요소들 사이에 원하는 구분자를 지정할 수있다. `\n` 을 구분자로 사용하면 텍스트 블록 사이에 줄바꿈이 들어가 **문단의 구조를 어느정도 살릴 수 있다.**
       -  웹 문서의 줄바꿈도 포함해서 읽기 때문에 공백과 줄바꿈이 혼재된 상태로 반환된다. `strip=True`로 설정하면 추출된 문자 앞뒤의 공백 문자들을 제거할 수있다.

In [39]:
!uv pip install beautifulsoup4 requests

Resolved 8 packages in 90ms
Prepared 2 packages in 81ms
Installed 2 packages in 16ms
 + beautifulsoup4==4.15.0
 + soupsieve==2.8.4


In [48]:
from bs4 import BeautifulSoup

html_txt = """<html>
<body>
<p><b>제목</b> <span>내용</span></p>
<p>다음문단</p>
<div>다음 내용</div>
</body>
</html>
"""

soup = BeautifulSoup(html_txt)

txt1 = soup.get_text() # 태그내의 text(content)만 추출한다.
txt2 = soup.get_text(strip=True) # tag사이의 공백(' ', 엔터, tab)을 제거.
txt3 = soup.get_text(strip=True, separator=",") 
# separator는 각 태그내의 text들을 지정한 구분자로 나눠서 반환.
print("-------기본--------")
print(txt1)
print("-------strip=True--------")
print(txt2)
print("-------strip=True, separator='\\n'--------")
print(txt3)

-------기본--------


제목 내용
다음문단
다음 내용



-------strip=True--------
제목내용다음문단다음 내용
-------strip=True, separator='\n'--------
제목,내용,다음문단,다음 내용


In [49]:
soup

<html>
<body>
<p><b>제목</b> <span>내용</span></p>
<p>다음문단</p>
<div>다음 내용</div>
</body>
</html>

In [ ]:
from bs4 import SoupStrainer

strainer = SoupStrainer(["p", "div"]) # <span>
soup2 = BeautifulSoup(html_txt, parse_only=strainer)
soup2

<p><b>제목</b> <span>내용</span></p><p>다음문단</p><div>다음 내용</div>

In [ ]:
!uv pip install lxml
# lxml 설치후에 kernel 재시작

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import os

os.environ['USER_AGENT'] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"

url = [
    "https://m.sports.naver.co  m/fifaworldcup2026/article/025/0003533131",
    "https://m.sports.naver.com/wbaseball/article/079/0004161711"
]

loader = WebBaseLoader(
    # web_path=url[1], # 1개 문서 조회
    web_path=url, # 여러개 문서 조회 list["url"들]
    # default_parser="lxml", # BeautifulSoup(문서str, parser(html.parser))
    bs_kwargs={ # BeautifulSoup() 생성할때 넣어줄 파라미터 설정.
        "parse_only":SoupStrainer(name="div", attrs={"class":"_article_content"})
    },
    bs_get_text_kwargs={ # get_text() 의 파라미터 설정
        "strip":True,
        "separator":"\n\n"
    }
)

In [69]:
docs = loader.load()
print(len(docs))
pprint(docs[0].metadata)
print(docs[0].page_content)

2
{'source': 'https://m.sports.naver.com/fifaworldcup2026/article/025/0003533131'}
옌스 카스트로프(오른쪽)가 25일 남아공전에서 타펠로 마세코의 슈팅을 막아보려 하고 있다. 그러나 공이 다리 사이를 지나가면서 실점으로 연결됐고, 이 득점은 결승점이 됐다. 연합뉴스

그토록 그리던 월드컵 데뷔전에서 뼈아픈 실점을 목격했다. 독일 혼혈 국가대표 옌스

카스트로프

는 아쉬움을 곱씹었다.

한국은 25일(한국시간) 멕시코 몬테레이 스타디움에서 열린

북중미 월드컵

조별리그 A조 남아프리카공화국과의 3차전에서 0-1로 졌다. 같은 날 멕시코가 체코를 3-0으로 꺾으면서 최종 순위는 멕시코 1위, 남아공 2위, 한국 3위가 됐다. 한국은 멕시코가 승리하면서 A조 3위를 지켰다. 다른 조 경기 결과를 지켜본 뒤 각조 3위와 승점, 골득실 등을 따져 32강 결선 토너먼트 진출 여부를 기다린다.

이날 한국은 답답한 흐름이 이어졌다. 전반 내내 이렇다 할 찬스를 잡지 못할 정도였다. 결국 홍명보 감독은 후반을 앞두고 공격수

손흥민

과 왼쪽 윙백 카스트로프를 투입했다. 그러나 이후에도 주도권을 가져오지 못했고, 결국 후반 18분 선취점을 내줬다. 체팡 모레미의 패스를 받은

타펠로 마세코

가 왼발로 때린 볼이 한국 골대 오른쪽 구석을 꿰뚫었다.

이때 마세코의 바로 앞을 지켰던 수비수가 카스트로프였다. 카스트로프는 몸을 날려 공을 막아보려 했지만, 그 사이로 슈팅이 지나가고 말았다.

이날이 월드컵 데뷔전이었던 카스트로프는 “월드컵 데뷔전을 치러 기쁘지만, 불행하게도 0-1로 졌다. 정말 아쉽다”고 고개를 숙였다. 이어 “아쉬운 결과지만, 이제 다른 조의 상황을 지켜봐야 한다. 32강으로 진출한다면 다음 경기에도 100% 집중하겠다”고 말했다.

한국인 어머니와 독일인 아버지 사이에서 태어난 카스트로프는 한국 축구 역사상 처음으로 해외 태생의 혼혈 국가대표로 발탁된 선수다. 지난해 9월 원정 평가전을 앞

#### RecursiveUrlLoader

- 주어진 URL에서 시작하여 그 페이지 안의 내부 링크를 재귀적으로 따라가며 여러 웹 문서를 자동 수집하여 로드한다.
  - 시작 url을 요청/페이지를 파싱 한 뒤에 `<a href>` 들을 수집하고 그 페이지들을 요청/페이지 파싱을 한다. 
- WebBaseLoader가 단일 페이지(단일 URL) 단위라면 RecursiveUrlLoader는 **웹 사이트 구조 전체를 크롤링하는 전용 수집기**에 가깝다.
  ```bash
  시작 URL
  ├─ 내부 링크 1
  │   ├─ 내부 링크 1-1
  │   └─ 내부 링크 1-2
  ├─ 내부 링크 2
  └─ 내부 링크 3
  ```
위 구조일때 무든 페이지를 재귀적으로 수집한다.
- 주요 파라미터
  - **url**: 시작 url
  - **max_depth**
    - 링크를 몇 단계 **깊이** 까지 따라갈지 제한
    - 사이트 폭주를 막기 위한 안전장치
      - **0**: 시작페이지만, **1**: 시작페이지 + 1차링크, **2**(기본값): 시작페이지 + 1차링크 + 2차링크
  - **exclude_dirs**: list[str]
    - 크롤링 제외 경로
    - ex) `exclude_dirs=['/login', 'signup']`
  - **prevent_outside**: bool
    - True: base_url 바깥 링크는 가져오지 않고 무시한다.
  - **base_url**: str
    - prevent_outside=True일 때 바깥링크의 기준. 없으면 `url`(시작 url)의 host가 된다. 
  - **extractor**
    - 문서 내용 추출 사용자 정의 함수
    - default는 응답 받은 페이지를 `BeautifulSoup(응답페이지).get_text()` 로 텍스트를 추출한다.
    - ````python
        def custom_extractor(html:str) ->str:
            # 웹 페이지 문서를 입력으로 받는다.
            soup = BeautifulSoup(html, 'lxml')
            return soup.select_one('article').get_text() # 원하는 항목을 추출해서 반환한다.
        
        loader = RecursiveUrlLoader(
            url=start_url,
            extractor=custom_extractor
        )    
    ```

In [ ]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader

start_url = "https://docs.python.org/3/"

def extractor(html:str) -> str:
    """RecursiveUrlLoader는 HTML문서를 그대로 반환.
    HTML 문서를 str으로 받아서 원하는 부분만 추출하는 callback 함수
    Args:
        html(str): HTML(웹) 문서
    Returns:
        str: html에서 body의 content(text)만 추출해서 반환.
    """
    soup = BeautifulSoup(html)
    body_element = soup.select_one("div.body") # python doc 사이트 - 내용이 <div class='body> 내에 위치
    return body_element.get_text(strip=True, separator="\n") \
           if body_element is not None else soup.get_text(strip=True, separator="\n")

loader = RecursiveUrlLoader(
    url=start_url,
    max_depth=2,  # 0: start_url, 1: start_url->link, 2: start_url->link->link
    prevent_outside=True,
    base_url=start_url,
    extractor=extractor
)

docs = loader.load()

C:\Users\Playdata\AppData\Local\Temp\ipykernel_24088\3263045571.py:14: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html)
c:\Documents\SKN31-inst\10_AI_Agent\.venv\Lib\site-packages\langchain_community\document_loaders\recursive_url_loader.py:44: XMLParsedAsHTMLWarning: It looks like you're using a

In [8]:
len(docs)

29

In [13]:
idx = 1
doc = docs[idx]
doc.metadata

{'source': 'https://docs.python.org/3/extending/index.html',
 'content_type': 'text/html',
 'title': 'Extending and Embedding the Python Interpreter — Python 3.14.6 documentation',
 'description': 'This document describes how to write modules in C or C++ to extend the Python interpreter with new modules. Those modules can not only define new functions but also new object types and their metho...',
 'language': 'en'}

In [4]:
# for doc in docs:
#     print(doc.metadata['source'])

In [14]:
print(doc.page_content)

Extending and Embedding the Python Interpreter
¶
This document describes how to write modules in C or C++ to extend the Python
interpreter with new modules.  Those modules can not only define new functions
but also new object types and their methods.  The document also describes how
to embed the Python interpreter in another application, for use as an extension
language.  Finally, it shows how to compile and link extension modules so that
they can be loaded dynamically (at run time) into the interpreter, if the
underlying operating system supports this feature.
This document assumes basic knowledge about Python.  For an informal
introduction to the language, see
The Python Tutorial
.
The Python Language Reference
gives a more formal definition of the language.
The Python Standard Library
documents
the existing object types, functions and modules (both built-in and written in
Python) that give the language its wide application range.
For a detailed description of the whole Python/C API,

### <del>ArxivLoader</del>

- arxiv api가 업데이트 되고 ArxivLoader는 그에 맞춰 업데이트가 되지 않아 ArxivLoader는 정상적으로 실행되지 않는다. **arxiv api 를 이용해서 검색 후 pdf를 다운로드 받는다.**
- arxiv API: https://github.com/lukasschwab/arxiv.py
- [arXiv-아카이브](https://arxiv.org/) 는 미국 코렐대학에서 운영하는 **무료 논문 저장소**로, 물리학, 수학, 컴퓨터 과학, 생물학, 금융, 경제 등 **과학, 금융 분야의 논문**들을 공유한다.
- 설치
  - `pip install arxiv`



In [15]:
!uv pip install arxiv

Resolved 7 packages in 197ms
Prepared 1 package in 20ms
Uninstalled 1 package in 13ms
Installed 2 packages in 67ms
 + arxiv==4.0.0
 - requests==2.34.2
 + requests==2.33.1


In [ ]:
# arxiv lib를 이용해서 arxiv.org의 논문을 검색해서 다운로드 or 논문의 주요 정보를 조회한다.
# RAG 문서로 arxiv의 논문들이 필요할 경우 이용할 수 있다.
import arxiv
# 검색관련 설정
search = arxiv.Search(
    query="Advanced RAG", # 검색어
    max_results=10, # 검색 논문 최대 개수
    sort_by=arxiv.SortCriterion.Relevance # 정렬기준.
)
# 정렬기준
# arxiv.SortCriterion
##      Relevance: query와 관련성이 높은(정확도) 순서
##      LastUpdatedDate: 논문이 마지막으로 수정 업데이트된 날짜.
##      SubmittedDate  : 논문이 처음 제출된 날짜

# 검색
client = arxiv.Client()
results = client.results(search)
print(type(results)) # Iterator

<class 'itertools.islice'>


In [ ]:
paper = next(results) # 첫번째(한개) 논문
print(type(paper))
# 아래 정보들 -> Document의 metadata
print(paper.title)# 논문 제목
print(paper.authors) # 논문 저자들 list[Author]
print(paper.authors[0].name) # 첫번째 저자의 이름
print(paper.categories) # 논문의 분야(카테고리)
print(paper.summary) #논문 요약 내용
print(paper.pdf_url) # 논문 PDF 파일의 url
print(paper.published) # 논문 발표 일시
print(paper.get_short_id()) # 논문 ID

<class 'arxiv.Result'>
MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for Multi-Hop Queries
[arxiv.Result.Author('Yixuan Tang'), arxiv.Result.Author('Yi Yang')]
Yixuan Tang
['cs.CL']
Retrieval-augmented generation (RAG) augments large language models (LLM) by retrieving relevant knowledge, showing promising potential in mitigating LLM hallucinations and enhancing response quality, thereby facilitating the great adoption of LLMs in practice. However, we find that existing RAG systems are inadequate in answering multi-hop queries, which require retrieving and reasoning over multiple pieces of supporting evidence. Furthermore, to our knowledge, no existing RAG benchmarking dataset focuses on multi-hop queries. In this paper, we develop a novel dataset, MultiHop-RAG, which consists of a knowledge base, a large collection of multi-hop queries, their ground-truth answers, and the associated supporting evidence. We detail the procedure of building the dataset, utilizing an English 

In [18]:
# 논문 pdf파일 다운로드
import os 
import requests

save_dir = "data/papers"
os.makedirs(save_dir, exist_ok=True)

resp = requests.get(paper.pdf_url)
if resp.status_code == 200:
    with open(os.path.join(save_dir, paper.get_short_id()+".pdf"), "wb") as fo:
        fo.write(resp.content)

In [3]:
# 논문 pdf 파일 다운로드 함수
import os 
import requests

def download_arxiv_paper(paper:arxiv.Result, dirpath:str):
    """검색결과(paper)를 다운로드 후 dirpath에 저장."""
    os.makedirs(dirpath, exist_ok=True)
    resp = requests.get(paper.pdf_url)
    if resp.status_code == 200:
        with open(os.path.join(dirpath, paper.get_short_id()+".pdf"), "wb") as fo:
            fo.write(resp.content)

In [20]:
save_dir = "data/papers"
for paper in results:
    download_arxiv_paper(paper, save_dir)

In [6]:
# arxiv에서 검색어의 논문을 조회해서 다운 받은 후에
#  Document에 page_content에는 논문 내용을 metadata에는 위의 정보를 넣어서 
#  list[Document] 를 반환하는 함수.
import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
def load_arxiv_docs(
        query: str, # 검색어
        top_k: int=10, # 최대 검색 개수
        dirpath: str="." # 논문저장할 디렉토리
) -> list[Document]:
    
    client = arxiv.Client()
    search = arxiv.Search(
        query=query, max_results=top_k, sort_by=arxiv.SortCriterion.Relevance
    )

    results = client.results(search)
    docs = [] # 생성한 Document(조회결과)를 담을 리스트
    for paper in results:
        # 다운로드
        download_arxiv_paper(paper, dirpath)
        # PDF 문서 load
        file_path = os.path.join(dirpath, paper.get_short_id()+".pdf")
        loader = PyPDFLoader(file_path, mode="single")
        doc:Document = loader.load()[0]  # list[Document]
        # 메타데이터는 paper의 정보로 변경.
        doc.metadata = {
            "title":paper.title,
            "authors":[a.name for a in paper.authors],
            "categories": paper.categories,
            "arxiv_url": paper.entry_id,   # 논문 페이지 url
            "pdf_url": paper.pdf_url
        }
        docs.append(doc)
    
    return docs

In [7]:
docs = load_arxiv_docs(query="transformers", top_k=20, dirpath="data/papers/transformers")

PdfReadError("Invalid Elementary Object starting with b'\\\\' @756840: b'ateVersion (2021.1) \\\\par /Author()/Title()/Subject()/Creator(LaTeX with hyperref'")
Multiple definitions in dictionary at byte 0xb8c76 for key /Author
Multiple definitions in dictionary at byte 0xb8c7e for key /Title


In [8]:
print(len(docs))

20


In [11]:
docs[10].metadata

{'title': 'PatchRot: A Self-Supervised Technique for Training Vision Transformers',
 'authors': ['Sachin Chhabra',
  'Prabal Bijoy Dutta',
  'Hemanth Venkateswara',
  'Baoxin Li'],
 'categories': ['cs.CV'],
 'arxiv_url': 'http://arxiv.org/abs/2210.15722v1',
 'pdf_url': 'https://arxiv.org/pdf/2210.15722v1'}

In [12]:
print(docs[10].page_content)

PatchRot: A Self-Supervised Technique for Training
Vision Transformers
Sachin Chhabra
Arizona State University
Tempe, AZ, USA
schhabr6@asu.edu
Prabal Bijoy Dutta
Arizona State University
Tempe, AZ, USA
pdutta6@asu.edu
Hemanth Venkateswara
Georgia State University
Atlanta, GA, USA
hvenkateswara@gsu.edu
Baoxin Li
Arizona State University
Tempe, AZ, USA
baoxin.li@asu.edu
Abstract
Vision transformers require a huge amount of labeled data to outperform convo-
lutional neural networks. However, labeling a huge dataset is a very expensive
process. Self-supervised learning techniques alleviate this problem by learning
features similar to supervised learning in an unsupervised way. In this paper, we
propose a self-supervised technique PatchRot that is crafted for vision transformers.
PatchRot rotates images and image patches and trains the network to predict the
rotation angles. The network learns to extract both global and local features from
an image. Our extensive experiments on different da

### Docling
- IBM Research에서 개발한 오픈소스 문서처리 도구로 다양한 종류의 문서를 구조화된 데이터로 변환해 생성형 AI에서 활용할 수있도록 지원한다.
- **주요기능**
  - PDF, DOCX, PPTX, XLSX, HTML, 이미지 등 여러 형식을 지원
  - PDF의 **페이지 레이아웃, 읽기 순서, 표 구조, 코드, 수식** 등을 분석하여 정확하게 읽어들인다.
  - OCR을 지원하여 스캔된 PDF나 이미지에서 텍스트를 추출할 수있다.
  - 읽어들인 내용을 markdown, html, json등 다양한 형식으로 출력해준다.
- 설치 : `pip install langchain-docling ipywidgets -qU` 
- 참조
  - docling 사이트: https://github.com/docling-project/docling
  - 랭체인-docling https://python.langchain.com/docs/integrations/document_loaders/docling/

In [1]:
# GPU가 있는 경우 -> torch 부터 gpu 버전으로 설치.
!uv pip install langchain-docling torch transformers accelerate

Resolved 119 packages in 6.54s
   Building antlr4-python3-runtime==4.9.3
   Building pylatexenc==2.10
 Downloaded shapely
 Downloaded docling-parse
 Downloaded pywin32
 Downloaded rapidocr
 Downloaded opencv-python
 Downloaded faker
 Downloaded saxonche
 Downloaded scipy
      Built pylatexenc==2.10
      Built antlr4-python3-runtime==4.9.3
Prepared 40 packages in 8.46s
Uninstalled 1 package in 23ms
Installed 45 packages in 1.22s
 + accelerate==1.14.0
 + antlr4-python3-runtime==4.9.3
 + colorlog==6.10.1
 + defusedxml==0.7.1
 + dill==0.4.1
 + doclang==0.7.0
 + docling==2.107.0
 + docling-core==2.85.0
 + docling-ibm-models==3.13.3
 + docling-parse==7.0.0
 + docling-slim==2.107.0
 + et-xmlfile==2.0.0
 + faker==40.23.0
 + jsonlines==4.0.0
 + jsonref==1.1.0
 + langchain-docling==2.0.0
 + latex2mathml==3.81.0
 + mail-parser==4.4.0
 + marko==2.2.3
 + mpire==2.10.2
 + multiprocess==0.70.19
 + omegaconf==2.3.1
 + opencv-python==4.13.0.92
 + openpyxl==3.1.5
 + pluggy==1.6.0
 + polyfactory==3.3.0

In [1]:
# Huggingface 로그인 - docling이 사용하는 모델은 로그인 후에 받을 수 있다.
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()

access_key = os.getenv("HUGGINGFACE_API_KEY")
login(access_key)

In [8]:
from langchain_docling import DoclingLoader 
from langchain_docling.loader import ExportType

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# 현재 torch와 docling이 사용하는 OCR Lib (RapidOCR) 호환성 문제 때문에 OCR 기능을 끄는 설정을 한다.
pdf_option = PdfPipelineOptions()
pdf_option.do_ocr = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pdf_option
        )
    }
)

path = ["data/papers/2409.03708v2.pdf", "data/papers/transformers/2209.03452v1.pdf"]
loader = DoclingLoader(
    file_path=path,
    export_type=ExportType.MARKDOWN,
    converter=converter
)

In [9]:
docs = loader.load()

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [10]:
len(docs)

2

In [11]:
doc = docs[1]
doc.metadata

{'source': 'data/papers/transformers/2209.03452v1.pdf'}

In [12]:
print(doc.page_content)

## AILAB-Udine@SMM4H'22: Limits of Transformers and BERT Ensembles

## Beatrice Portelli

portelli.beatrice@spes.uniud.it University of Udine, Italy University of Naples Federico II , Italy

## Simone Scaboro

scaboro.simone@spes.uniud.it University of Udine, Italy

## Enrico Santus

esantus@gmail.com DSIG - Bayer Pharmaceuticals, New Jersey, USA

## Abstract

This paper describes the models developed by the AILAB-Udine team for the SMM4H'22 Shared Task. We explored the limits of Transformer based models on text classification, entity extraction and entity normalization, tackling Tasks 1, 2, 5, 6 and 10. The main takeaways we got from participating in different tasks are: the overwhelming positive effects of combining different architectures when using ensemble learning, and the great potential of generative models for term normalization.

## 1 Introduction

Transformer-based models are the backbone of state-of-the-art solutions for a lot of NLP tasks. The real strength of these models

In [7]:
from IPython.display import Markdown

Markdown(doc.page_content)

## RAG based Question-Answering for Contextual Response Prediction System

Sriram Veturi ∗ Saurabh Vaichal ∗

sriram\_veturi@homedepot.com saurabh\_s\_vaichal@homedepot.com The Home Depot Atlanta, Georgia, USA

Nafis Irtiza Tripto ‡

The Pennsylvania State University

USA nit5154@psu.edu

reshma\_lal\_jagadheesh@homedepot.com Reshma Lal Jagadheesh † The Home Depot Atlanta, Georgia, USA

## Nian Yan

The Home Depot Atlanta, Georgia, USA nian\_yan@homedepot.com

## CCS Concepts

- Computing methodologies → Machine learning .

## Keywords

Retrieval Augmented Generation, Response Prediction System, Question Answering System, Contact Center Agents, Automated Hallucination Measurement, Hallucination Reduction, Retrieval Strategies, Optimal Retriever Threshold, ScaNN, Embedding Strategies, Contextual Relevance, Specificity, Completeness, Hallucination Rate, Missing Rate, Human Evaluation of RAG Versus Traditional Seq-2-Seq models, RAG Deployment.

## ACMReference Format:

Sriram Veturi, Saurabh Vaichal, Reshma Lal Jagadheesh, Nafis Irtiza Tripto, and Nian Yan. 2024. RAG based Question-Answering for Contextual Response Prediction System. In Proceedings of CIKM 2024 (1st Workshop on GenAI and RAG Systems for Enterprise). ACM, New York, NY, USA, 10 pages. https://doi.org/10.1145/nnnnnnn.nnnnnnn

## 1 Introduction

With the advent of ChatGPT and similar tools in mainstream media, Large Language Models (LLMs) have emerged as the standard solution for addressing a wide range of language understanding tasks. However, they can generate incorrect or biased information [34], as their responses are based on patterns learned from data that may not always contain necessary knowledge in a close domain. To address this issue, Retrieval Augmented Generation (RAG) [20] is commonly used to ground LLMs in factual information. The RAG architecture processes user input by first retrieving a set of documents similar to the query, which the language model then uses to generate a final prediction. While RAG-based architectures have been successful in various open-domain question answering (Q/A) tasks [15, 30, 49], limited research has explored their scaling dynamics in real conversational scenarios. Therefore, our research is one of the pioneering efforts in exploring the feasibility of an RAGbased approach for developing a knowledge-grounded response prediction system specifically tailored for the contact center of a major retail company.

## Abstract

Large Language Models (LLMs) have shown versatility in various Natural Language Processing (NLP) tasks, including their potential as effective question-answering systems. However, to provide precise and relevant information in response to specific customer queries in industry settings, LLMs require access to a comprehensive knowledge base to avoid hallucinations. Retrieval Augmented Generation (RAG) emerges as a promising technique to address this challenge. Yet, developing an accurate question-answering framework for real-world applications using RAG entails several challenges: 1) data availability issues, 2) evaluating the quality of generated content, and 3) the costly nature of human evaluation. In this paper, we introduce an end-to-end framework that employs LLMs with RAG capabilities for industry use cases. Given a customer query, the proposed system retrieves relevant knowledge documents and leverages them, along with previous chat history, to generate response suggestions for customer service agents in the contact centers of a major retail company. Through comprehensive automated and human evaluations, we show that this solution outperforms the current BERT-based algorithms in accuracy and relevance. Our findings suggest that RAG-based LLMs can be an excellent support to human customer service representatives by lightening their workload.

∗ Both first authors contributed equally to this research.

† Work completed during employment at the Home Depot

‡ Work completed during internship at The Home Depot

Permission to make digital or hard copies of all or part of this work for personal or classroom use is granted without fee provided that copies are not made or distributed for profit or commercial advantage and that copies bear this notice and the full citation on the first page. Copyrights for components of this work owned by others than the author(s) must be honored. Abstracting with credit is permitted. To copy otherwise, or republish, to post on servers or to redistribute to lists, requires prior specific permission and/or a fee. Request permissions from permissions@acm.org.

1st Workshop on GenAI and RAG Systems for Enterprise, October 24, 2024, Boise, Idaho, USA

© 2024 Copyright held by the owner/author(s). Publication rights licensed to ACM.

ACM ISBN 978-x-xxxx-xxxx-x/YY/MM

https://doi.org/10.1145/nnnnnnn.nnnnnnn

Figure 1: Example of the Response Prediction System. (A): For a valid query, the system retrieves the relevant document and proposes the appropriate responses from where the agent choose. (B): For an out-of-domain query, it guides the user to ask a relevant question.

LLMs have recently been widely adopted across various industries, particularly in contact centers, to enhance chatbot development and agent-facing automation [8, 12, 37]. A prime example is the Response Prediction System (RPS), an agent-assist solution that generates contextually relevant responses, enabling agents to efficiently address customer queries with a single click. This boosts productivity, improves customer experience, and streamlines communication processes. In industry settings, the focus is on generating accurate, contextually appropriate responses with minimal latency. Therefore, RAG-based responses, grounded in company policies, deliver swift and accurate resolutions to customer issues. Figure 1 demonstrates a possible example of RPS in real settings, where the agent can directly utilize the generated response with a single click.

However, implementing RAG for industry-specific use cases to assist human agents in generating valid responses involves several architectural decisions that can affect performance and viability. The retrieval style can be integrated into both encoder-decoder [16, 44]) and decoder-only models [5, 19, 26, 27], with various embedding and prompting techniques influencing the final LLM output. In contact centers, where the risk of hallucinations is high and can critically impact business performance, ReAct (Reason+Act) [42] prompts can help mitigate issues. Therefore, our research focuses on developing an optimal RAG based knowledge-grounded RPS for a major retail company's contact center. To ensure response accuracy, we also conduct thorough evaluations with human evaluators and automated measures, comparing RAG-based responses to human ground truth and the existing BERT-based system (Figure 2 shows an overview of traditional customer care scenario with existing and proposed system). In short, we answer the following research questions.

- (1) RQ1: What are the effects of different embedding techniques, retrieval strategies, and prompting methods on RAG performance?
- (2) RQ2: Do RAG-based responses provide greater assistance to human agents compared to the existing BERT-based system?
- (3) RQ3: Can the ReAct (Reason+Act) prompting improve factual accuracy and reduce hallucinations in LLM in real-time settings?

Our findings demonstrate an overall improvement over the existing system by suggesting more accurate and relevant responses, highlighting the potential of RAG LLM as an excellent choice for customer care automation.

## 2 Related Work

RAG architecture: RAG has emerged as a promising solution by incorporating knowledge from external databases to overcome the hallucination, outdated knowledge, transparency issues for LLMs [5, 17, 19, 20, 43]. Traditional RAG, popularized after the adoption of ChatGPT, follows a simple process of indexing, retrieval, and generation [14] . Despite advancements in Advanced and Modular RAG, Traditional RAG remains popular in the industry due to its ease of development, integration, and quicker speed to market [21]. The core components of Traditional RAG include Retriever, Generator, and Augmentation Method, with research focusing on improving semantic representation [9, 48], query alignment [35], and integration with LLMs [16, 28, 45], which motivate our RQ1 to find the optimum setup in this specific use case of RPS in the contact center.

RAG LLM for question answering: Several open-domain questionanswering (Q/A) tasks have been completed by RAG-based architectures efficiently [15, 30, 49]. With the advent of LLMs in recent periods, multiple studies also focus on utilizing LLMs for customer assistance, specifically in recommendations [13, 41] and dialogue generation [29, 47]. Recent work by [2] proposes augmenting LLMs with user-specific context from search engine interaction histories to personalize outputs, leveraging entity-centric knowledge stores derived from users' web activities. Similarly, [40] introduces a customer service question-answering approach integrating RAG with a knowledge graph (KG) constructed from historical issue data. Therefore, our study is motivated by these prior researches to integrate RAG as a retrieval tool and utilize LLM to generate responses to answer customer queries.

## 3 Methodology

To implement an end-to-end RAG framework with LLM, first, it is essential to create a comprehensive dataset comprising relevant question-answer pairs along with corresponding knowledge documents. Next, design choices for specific components of the RAG and LLM architecture must be finalized. Finally, the model should be thoroughly evaluated and refined before being deployed into production.

## 3.1 Phase I: Data Preparation

An ideal golden dataset for evaluating RAG architecture (Figure 3) should include:

- (1) Domain-specific questions (previous queries) with their corresponding grounded responses.
- (2) Relevant knowledge base (KB) articles (company documents) containing the policies that determine answers to specific queries.

Figure 2: Overview of the systems: (A) Agents respond to queries by manually searching for relevant documents, (B) The existing BERT-based system, which extracts relevant Q/A pairs from the given query and provides suggested answers to the agents, (C) The proposed RAG LLM system, where the LLM retrieves relevant KB articles (if necessary) and generates answers based on the query and the retrieved articles.

Figure 3: End to end RAG LLM framework

- (3) Out-of-domain questions to ensure the LLM can handle generic queries without hallucinating and can guide customers to provide relevant queries.

Table 1: Total number and average length (in terms of word count) for the KB articles and various Q/A pairs for the RAG implementation

| Source                                            |   Total # | length (query)               | length (response)            |
|---------------------------------------------------|-----------|------------------------------|------------------------------|
| KB articles                                       |      1205 | Avg. document length: 134.25 | Avg. document length: 134.25 |
| In domain Q/A (generated by LLM from KB articles) |      4785 | 10.7                         | 33.59                        |
| In domain Q/A (sampled from previous history)     |      3000 | 9.58                         | 28.81                        |
| Out domain Q/A (sampled from MS-MARCO)            |      3660 | 5.73                         | 5                            |

To create a robust test set (Table 1 for details), we utilize LLM to generate both relevant question-answer pairs from the company's KB articles (refer to Section A in the Appendix for prompts). Additionally, we supplement these relevant pairs with samples from previous queries &amp; responses in the contact center with out-ofdomain questions by sampling from open-source datasets such as MS-MARCO [3].

## 3.2 Phase II: RAG

The main components of the RAG architecture are the Retriever and the Generator LLM. We evaluate various strategies for each component and finalize our choices for production. Our findings are validated through experiments with several open-domain questionanswer datasets, including MARCO [3], SQuAD [23], and TriviaQA [18] (details in Appendix). They present a comparable level of question-answering challenges where answers can be derived from the retrieved knowledge base.

Embedding Strategy: The best embedding strategy ensures high performance of the retriever and affects downstream tasks like response generation. We compare the Universal Sentence Encoder (USE) embeddings [6], Google's Vertex AI embedding model textembedding-gecko@001 [ ? ], and SBERT-all-mpnet-base-v2 [24] from the sentence-transformers collection.

Retrieval strategies: By retrieving relevant passages from a large corpus of KB articles, the model gains crucial contextual information, enhancing response accuracy and coherence. We specifically consider ScaNN [25] for its efficiency in handling large-scale datasets and KNN HNSW [22] for its efficient memory usage as retrieval strategies in our study. Additionally, we tested different retrieval thresholds to ensure incorrect documents are not retrieved and passed to the LLM for response generation.

LLM for generation: Once the best embedding strategy, retrieval technique, and retrieval thresholds are identified, we test different prompting techniques to ensure that LLMs generate grounded factual responses. We utilize PaLM2 foundation models (text-bison, text-unicorn) [1] for text generation across all tasks, as they offer a clear path to production in terms of enterprise licenses and security requirements with Google's models, compared to other available LLMs at the time of our research.

The best model from Phase 2, incorporating the optimal embedding, retrieval, and prompting techniques, is packaged with relevant KB articles and deployed on a cloud Virtual Machine. For real-time usage, an endpoint is created that takes a customer query and the conversation context as input, generating response suggestions as output.

## 4 Results and Findings

First, we optimize RAG setup for our use cases, then evaluate LLM responses using automated metrics and human evaluations. Finally, we assess if prompting or ReAct strategies can improve real-world performance to an acceptable level.

## 4.1 Retrieval evaluation

Best setting for RAG. :

We assessed retriever efficiency using the "Recall at K" ( 𝑅 @ 𝑘 ) metric, where 𝐾 represents the top 1, 3, 5, or 10 documents retrieved, measuring how well the retriever retrieves relevant documents. The Vertex AI - textembedding-gecko@001 (768) embedding, paired with ScaNN retrieval, yielded the best outcomes. Overall, ScaNN generally outperformed KNN HNSW in most cases due to its efficient handling of large-scale datasets and superior retrieval accuracy through quantization and re-ranking techniques [7], so we include only the ScaNN results in Table 2. Similarly, Vertex AI embeddings surpassed Sentence BERT and USE due to its superior ability to capture complex semantic relationships tailored for large-scale industry applications.

Retrieval Threshold: For out-of-domain or trivial customer queries like "Hello" or "Bye," document retrieval is unnecessary, as shown by 98.59% of retrieved articles having a cosine similarity score below 0.7. In contrast, 88.96% of articles retrieved for relevant company data questions scored above 0.7 (Figure 4). This suggests that setting the retrieval threshold at 0.7 effectively determines when retrieval is needed, thereby enhancing response generation efficiency.

| Embedding   |   Embedding size | R@1    | R@3    | R@5    |
|-------------|------------------|--------|--------|--------|
| USE         |              512 | -      | -      | -      |
| SBERT       |              768 | +15.36 | +9.42  | +8.22  |
| Vertex AI   |              768 | +21.55 | +13.87 | +11.85 |

Table 2: Performance of different embedding technique in the company data for ScaNN. Performance is shown as the (%) of improvement wrt the lowest performing embedding (USE in this scenario)

Figure 4: Cosine similarity score between query and ScaNN retrieved Document; retrieval threshold(0.7)

Table 3: Comparision between RAG based responses and existing BERT-based ones for automated evaluations. Values indicate the difference in percentage (%) as average of all samples

| Metric              |   Improvement |
|---------------------|---------------|
| Accuracy            |        +10.15 |
| Hallucination       |         -4.76 |
| Missing rate        |         -5.43 |
| AlignScore          |          +5.6 |
| Semantic similarity |        +20.01 |
| AI-generated        |        -40.17 |

## 4.2 Response Prediction System Evaluation

To develop an effective Response Generation System (RPS), we conducted a comprehensive evaluation comparing RAG LLM-based responses with a current BERT-based algorithm. Using 1,000 real contact center chat transcripts (PII and PCI compliant), comprising over 5,000 messages, we analyzed customer queries, human agent responses, RAG LLM suggestions, BERT-based suggestions, and retrieved knowledge base documents to assess quality, consistency, and factuality through automated measures and human evaluations.

4.2.1 Automated evaluations. We utilize the following evaluation techniques, with Table 3 illustrating our RAG LLM-based technique's performance against the current BERT-based system.

Accuracy, Hallucination and Missing rate evaluation. In a questionanswer system, a response to each query can generate one of three types of responses: accurate (correctly answers the question), hallucinate (incorrect answer), or missing (no answer generated). Therefore, our approach, inspired by [32] which provides 98% agreement with human judgments, utilizes an LLM-based method. We employ ChatGPT-3.5-turbo as our evaluator LLM. We prompted the LLM with a query, generated response, and original human response, categorizing the LLM's responses as "correct" for factual and semantic alignment, "incorrect" for mismatches, and "unsure" for semantic challenges. Evaluation includes Accuracy (correct responses), Hallucination Rate (incorrect responses), and Missing Rate (unsure responses) metrics as the proportion of corresponding responses. Overall, RAG LLM improves accuracy by reducing hallucinations and missing rates compared to BERT responses.

AlignScore: To ensure response alignment with KB articles, we use AlignScore [46] to measure information consistency. Evaluating RAG LLM and BERT-based models on utterances with relevant KB article retrieval by RAG, RAG LLM shows a statistically significant 5.6% improvement via Student's t-test. This enhancement derives from integrating retrieved documents as prompts for LLM responses, whereas BERT relies on query-answer pairs in its training dataset.

Semantic similarity: To ensure usability by human agents, coherence between generated and original human responses is crucial. We measure semantic similarity using LongFormer embeddings [4], calculating cosine similarity between generated and original human responses for both models. RAG LLM exhibits an average 20% higher similarity, a statistically significant improvement.

Human touch: Customer service are typically preferred to be handled by humans [11, 38], emphasizing the importance of generating human-like responses. We use the AI text detector GPTZero [33], with a 99.05% true positive rate for human responses in our dataset, to evaluate response naturalness. Assessing AI percentage (utterances identified as AI-generated), the BERT-based system, which selects responses from human-generated options, sounds more human.

4.2.2 Human evaluations. Our method aims to support rather than replace humans through a human-in-the-loop approach. We thoroughly evaluate the quality of RAG LLM and BERT responses using human annotators. Each response is assessed against several criteria, and the average score is computed from all annotators' evaluations. Evaluation metrics were grouped into three main categories:

Human Preference Score: Following the classical approach of which version humans prefer most [31, 39], we evaluated which model's responses-"BERT" or "RAG"-were preferred by human evaluators.

Quantitative Metrics: Similar to [32], we evaluated factual accuracy (based on human judgment of 'correct,' 'incorrect,' or 'unsure'). Accuracy, Hallucination, and Missing rates were calculated as the number of correct, incorrect, and unsure responses divided by the total number of responses evaluated, respectively.

Qualitative Metrics:

- (1) Contextual Relevance: Assessed whether the predicted responses were appropriate and in line with the context of the conversation.
- (2) Completeness: Checked if the predicted responses were fully-formed and could be used as complete answers by the agents in specific parts of the conversation.
- (3) Specificity: Determined whether the predicted responses were tailored to the specific conversation or were too general. Human annotators scored these metrics on a scale of 0 (lowest) to 2 (highest).

Table 4: Human evaluation comparison (% diff.) between RAG and existing BERT-based ones.

| Metric               |   Improvement |
|----------------------|---------------|
| Contextual Relevance |        +48.14 |
| Specificity          |        +97.97 |
| Completeness         |        +70.15 |
| Accuracy             |        +45.69 |
| Hallucination Rate   |        -27.49 |
| Missing Rate         |        -70.02 |
| Preference           |       +200.61 |

Table 5: Comparison (% diff.) of ReAct RAG (with different values of k), CoVe and CoTP performance with respect to baseline on company data.

|   K | Strategy   |   Accuracy |   Hallucination Rate |   Missing Rate |
|-----|------------|------------|----------------------|----------------|
|   1 | ReAct      |      -2.13 |               +51.40 |         -34.38 |
|   3 | ReAct      |      +7.08 |               -13.48 |         -19.38 |
|   3 | CoVe       |     -43.65 |               +27.43 |         +11.35 |
|   3 | CoTP       |      -3.45 |                +1.33 |          +1.98 |

The results, as detailed in the Table 4, Responses generated by the RAG model demonstrated a 45% improvement in factual accuracy and a 27% decrease in the rate of hallucinations compared to the existing model. Moreover, the human evaluators favored responses from the RAG model over the current production model 75% of the times.

The Response Prediction System was deployed using Flask, a standard micro web framework, and Gunicorn, chosen for its performance, flexibility, and simplicity in production system configuration. The API receives customer queries as input and provides answers as output. The API was thoroughly load tested using Locust, an open-source performance/load testing tool for HTTP and other protocols, to ensure it meets real-time latency requirements in a production setting. Finally, the API was integrated with the Agent Workspace UI to deliver predictions to Contact Center agents, assisting customers in real-time.

## 4.3 Evaluation for ReAct and prompting techniques

Experiments with ReAct. To answer our third RQ, we utilized ReAct Tools to determine when to activate the information retrieval component within the RAG framework, while maintaining the same retrieval, embeddings, and generation strategies. We evaluated two scenarios: "RAG with ReAct" and "RAG without ReAct," with K = 3. As shown in Table 5. While ReAct improved the accuracy by 7% and reduced hallucination by 13.5%, it resulted in slower performance 6, making it inconvenient in real-time conversation.

Table 6: Latency Comparison (seconds) between ReAct RAG and non-ReAct RAG based on 10000 queries

| RAG Strategy   |   95th Percentile |   99th Percentile |
|----------------|-------------------|-------------------|
| reAct          |            4.0942 |            6.2084 |
| non-reAct      |            0.8850 |            1.1678 |

Prompting Techniques Experiments. We evaluated Chain of Verification (CoVe) [10] and Chain of Thought Prompting (CoTP) [36] to improve factual accuracy and reduce hallucinations. However, both techniques are time-consuming, requiring multiple LLM calls per query, and did not show significant improvements for the Company data. CoVe was 43% less accurate and CoTP was 3% less accurate (see Table 5). Therefore, we decided against using these prompting techniques.

## 5 Conclusion

In this study, we demonstrate the practical challenges of implementing a RAG-based Response Prediction System in an industry setting. We evaluated various retrieval and embedding strategies combined with different prompting techniques to identify the best combinations for different use cases. Our evaluations show that retrieving relevant knowledge base articles and generating responses from LLMs can be more contextually relevant and accurate than BERT responses, which choose from the most relevant query-answer pairs. We also highlight that ReAct and advanced prompting techniques may not be practical for industry settings due to latency issues. Overall, our approach indicates that implementing RAG-based LLM response generation for contact centers is feasible and can effectively aid humans, reducing their workload. In the future, we plan to advance our work in three directions. Firstly, we aim to evaluate other LLMs. Secondly, we will test if query rewriting and reformulation can improve retrieval performance. Lastly, we intend to explore advanced RAG approaches to integrate knowledge bases from various sources.

## Limitations

Despite ongoing significant research, LLMs remain unpredictable. Though this paper showcases work on grounding the generated responses, LLMs to a certain degree are still capable of generating inaccurate information based on their learnt parametric memory. This work also does not focus on other LLM issues such as context length constraints, prompt injections and quality of Knowledge base data. Addressing other open challenge such as biases along with their ethical consideration are also not considered in the scope of this paper. The paper also does not address or evaluate RAG for multilingual data sources.

## Ethics Statement

The data set used for training and validation of LLMs in this paper do not have any unbalanced views or opinions of individuals that might bias the generated response. The LLMs are still capable of generating inaccurate responses based their parametric memory even when relevant contextual information might be provided. Filtering toxic responses and prompt injections are also not considered in this evaluation.

## References

- [1] Rohan Anil, Andrew M. Dai, Orhan Firat, Melvin Johnson, Dmitry Lepikhin, Alexandre Passos, Siamak Shakeri, Emanuel Taropa, Paige Bailey, Zhifeng Chen, Eric Chu, Jonathan H. Clark, Laurent El Shafey, Yanping Huang, Kathy MeierHellstern, Gaurav Mishra, Erica Moreira, Mark Omernick, Kevin Robinson, Sebastian Ruder, Yi Tay, Kefan Xiao, Yuanzhong Xu, Yujing Zhang, Gustavo Hernandez Abrego, Junwhan Ahn, Jacob Austin, Paul Barham, Jan Botha, James Bradbury, Siddhartha Brahma, Kevin Brooks, Michele Catasta, Yong Cheng, Colin Cherry, Christopher A. Choquette-Choo, Aakanksha Chowdhery, Clément Crepy, Shachi Dave, Mostafa Dehghani, Sunipa Dev, Jacob Devlin, Mark Díaz, Nan Du, Ethan Dyer, Vlad Feinberg, Fangxiaoyu Feng, Vlad Fienber, Markus Freitag, Xavier Garcia, Sebastian Gehrmann, Lucas Gonzalez, Guy Gur-Ari, Steven Hand, Hadi Hashemi, Le Hou, Joshua Howland, Andrea Hu, Jeffrey Hui, Jeremy Hurwitz, Michael Isard, Abe Ittycheriah, Matthew Jagielski, Wenhao Jia, Kathleen Kenealy, Maxim Krikun, Sneha Kudugunta, Chang Lan, Katherine Lee, Benjamin Lee, Eric Li, Music Li, Wei Li, YaGuang Li, Jian Li, Hyeontaek Lim, Hanzhao Lin, Zhongtao Liu, Frederick Liu, Marcello Maggioni, Aroma Mahendru, Joshua Maynez, Vedant Misra, Maysam Moussalem, Zachary Nado, John Nham, Eric Ni, Andrew Nystrom, Alicia Parrish, Marie Pellat, Martin Polacek, Alex Polozov, Reiner Pope, Siyuan Qiao, Emily Reif, Bryan Richter, Parker Riley, Alex Castro Ros, Aurko Roy, Brennan Saeta, Rajkumar Samuel, Renee Shelby, Ambrose Slone, Daniel Smilkov, David R. So, Daniel Sohn, Simon Tokumine, Dasha Valter, Vijay Vasudevan, Kiran Vodrahalli, Xuezhi Wang, Pidong Wang, Zirui Wang, Tao Wang, John Wieting, Yuhuai Wu, Kelvin Xu, Yunhan Xu, Linting Xue, Pengcheng Yin, Jiahui Yu, Qiao Zhang, Steven Zheng, Ce Zheng, Weikang Zhou, Denny Zhou, Slav Petrov, and Yonghui Wu. 2023. PaLM 2 Technical Report. arXiv:2305.10403 [cs.CL]
- [2] Jinheon Baek, Nirupama Chandrasekaran, Silviu Cucerzan, Allen Herring, and Sujay Kumar Jauhar. 2024. Knowledge-augmented large language models for personalized contextual query suggestion. In Proceedings of the ACM on Web Conference 2024 . 3355-3366.
- [3] Payal Bajaj, Daniel Campos, Nick Craswell, Li Deng, Jianfeng Gao, Xiaodong Liu, Rangan Majumder, Andrew McNamara, Bhaskar Mitra, Tri Nguyen, Mir Rosenberg, Xia Song, Alina Stoica, Saurabh Tiwary, and Tong Wang. 2018. MS MARCO: A Human Generated MAchine Reading COmprehension Dataset. arXiv:1611.09268 [cs.CL]
- [4] Iz Beltagy, Matthew E Peters, and Arman Cohan. 2020. Longformer: The longdocument transformer. arXiv preprint arXiv:2004.05150 (2020).
- [5] Sebastian Borgeaud, Arthur Mensch, Jordan Hoffmann, Trevor Cai, Eliza Rutherford, Katie Millican, George van den Driessche, Jean-Baptiste Lespiau, Bogdan Damoc, Aidan Clark, Diego de Las Casas, Aurelia Guy, Jacob Menick, Roman Ring, Tom Hennigan, Saffron Huang, Loren Maggiore, Chris Jones, Albin Cassirer, Andy Brock, Michela Paganini, Geoffrey Irving, Oriol Vinyals, Simon Osindero, Karen Simonyan, Jack W. Rae, Erich Elsen, and Laurent Sifre. 2022. Improving language models by retrieving from trillions of tokens. arXiv:2112.04426 [cs.CL]
- [6] Daniel Cer, Yinfei Yang, Sheng yi Kong, Nan Hua, Nicole Limtiaco, Rhomni St. John, Noah Constant, Mario Guajardo-Cespedes, Steve Yuan, Chris Tar, YunHsuan Sung, Brian Strope, and Ray Kurzweil. 2018. Universal Sentence Encoder. arXiv:1803.11175 [cs.CL]
- [7] Wei-Cheng Chang, Jyun-Yu Jiang, Jiong Zhang, Mutasem Al-Darabsah, Choon Hui Teo, Cho-Jui Hsieh, Hsiang-Fu Yu, and SVN Vishwanathan. 2024. PEFA: ParamEter-Free Adapters for large-scale embedding-based retrieval models. In Proceedings of the 17th ACM International Conference on Web Search and Data Mining . 77-86.
- [8] Wei-Lin Chiang, Lianmin Zheng, Ying Sheng, Anastasios Nikolas Angelopoulos, Tianle Li, Dacheng Li, Hao Zhang, Banghua Zhu, Michael Jordan, Joseph E Gonzalez, et al. 2024. Chatbot arena: An open platform for evaluating llms by human preference. arXiv preprint arXiv:2403.04132 (2024).
- [9] Zhuyun Dai, Vincent Y. Zhao, Ji Ma, Yi Luan, Jianmo Ni, Jing Lu, Anton Bakalov, Kelvin Guu, Keith B. Hall, and Ming-Wei Chang. 2022. Promptagator: Few-shot Dense Retrieval From 8 Examples. arXiv:2209.11755 [cs.CL]
- [10] Shehzaad Dhuliawala, Mojtaba Komeili, Jing Xu, Roberta Raileanu, Xian Li, Asli Celikyilmaz, and Jason Weston. 2023. Chain-of-Verification Reduces Hallucination in Large Language Models. arXiv:2309.11495 [cs.CL]
- [11] Teresa Fernandes and Elisabete Oliveira. 2021. Understanding consumers' acceptance of automated technologies in service encounters: Drivers of digital voice assistants adoption. Journal of Business Research 122 (2021), 180-191.
- [12] Samuel Kernan Freire, Chaofan Wang, and Evangelos Niforatos. 2024. Chatbots in knowledge-intensive contexts: Comparing intent and llm-based systems. arXiv preprint arXiv:2402.04955 (2024).
- [13] Luke Friedman, Sameer Ahuja, David Allen, Zhenning Tan, Hakim Sidahmed, Changbo Long, Jun Xie, Gabriel Schubiner, Ajay Patel, Harsh Lara, et al. 2023. Leveraging large language models in conversational recommender systems. arXiv preprint arXiv:2305.07961 (2023).
- [14] Yunfan Gao, Yun Xiong, Xinyu Gao, Kangxiang Jia, Jinliu Pan, Yuxi Bi, Yi Dai, Jiawei Sun, Qianyu Guo, Meng Wang, and Haofen Wang. 2024. Retrieval-Augmented Generation for Large Language Models: A Survey. arXiv:2312.10997 [cs.CL]

- [15] Gautier Izacard and Edouard Grave. 2020. Leveraging passage retrieval with generative models for open domain question answering. arXiv preprint arXiv:2007.01282 (2020).
- [16] Gautier Izacard, Patrick Lewis, Maria Lomeli, Lucas Hosseini, Fabio Petroni, Timo Schick, Jane Dwivedi-Yu, Armand Joulin, Sebastian Riedel, and Edouard Grave. 2022. Atlas: Few-shot Learning with Retrieval Augmented Language Models. arXiv:2208.03299 [cs.CL]
- [17] Gautier Izacard, Patrick Lewis, Maria Lomeli, Lucas Hosseini, Fabio Petroni, Timo Schick, Jane Dwivedi-Yu, Armand Joulin, Sebastian Riedel, and Edouard Grave. 2023. Atlas: Few-shot Learning with Retrieval Augmented Language Models. Journal of Machine Learning Research 24, 251 (2023), 1-43. http://jmlr.org/papers/ v24/23-0037.html
- [18] Mandar Joshi, Eunsol Choi, Daniel Weld, and Luke Zettlemoyer. 2017. TriviaQA: A Large Scale Distantly Supervised Challenge Dataset for Reading Comprehension. In Proceedings of the 55th Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers) , Regina Barzilay and Min-Yen Kan (Eds.). Association for Computational Linguistics, Vancouver, Canada, 1601-1611. https: //doi.org/10.18653/v1/P17-1147
- [19] Urvashi Khandelwal, Omer Levy, Dan Jurafsky, Luke Zettlemoyer, and Mike Lewis. 2020. Generalization through Memorization: Nearest Neighbor Language Models. arXiv:1911.00172 [cs.CL]
- [20] Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, Vladimir Karpukhin, Naman Goyal, Heinrich Küttler, Mike Lewis, Wen tau Yih, Tim Rocktäschel, Sebastian Riedel, and Douwe Kiela. 2021. Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. arXiv:2005.11401 [cs.CL]
- [21] Nelson F. Liu, Kevin Lin, John Hewitt, Ashwin Paranjape, Michele Bevilacqua, Fabio Petroni, and Percy Liang. 2023. Lost in the Middle: How Language Models Use Long Contexts. arXiv:2307.03172 [cs.CL]
- [22] Yu. A. Malkov and D. A. Yashunin. 2018. Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs. arXiv:1603.09320 [cs.DS]
- [23] Pranav Rajpurkar, Jian Zhang, Konstantin Lopyrev, and Percy Liang. 2016. SQuAD: 100,000+ Questions for Machine Comprehension of Text. In Proceedings of the 2016 Conference on Empirical Methods in Natural Language Processing , Jian Su, Kevin Duh, and Xavier Carreras (Eds.). Association for Computational Linguistics, Austin, Texas, 2383-2392. https://doi.org/10.18653/v1/D16-1264
- [24] Nils Reimers and Iryna Gurevych. 2019. Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. arXiv:1908.10084 [cs.CL]
- [25] Google Research. 2020. ScaNN: Efficient Vector Similarity Search. https://github. com/google-research/google-research/tree/master/scann.
- [26] Ohad Rubin, Jonathan Herzig, and Jonathan Berant. 2022. Learning To Retrieve Prompts for In-Context Learning. arXiv:2112.08633 [cs.CL]
- [27] Weijia Shi, Julian Michael, Suchin Gururangan, and Luke Zettlemoyer. 2022. kNN-Prompt: Nearest Neighbor Zero-Shot Inference. arXiv:2205.13792 [cs.CL]
- [28] Weijia Shi, Sewon Min, Michihiro Yasunaga, Minjoon Seo, Rich James, Mike Lewis, Luke Zettlemoyer, and Wen tau Yih. 2023. REPLUG: Retrieval-Augmented Black-Box Language Models. arXiv:2301.12652 [cs.CL]
- [29] Kurt Shuster, Jing Xu, Mojtaba Komeili, Da Ju, Eric Michael Smith, Stephen Roller, Megan Ung, Moya Chen, Kushal Arora, Joshua Lane, et al. 2022. Blenderbot 3: a deployed conversational agent that continually learns to responsibly engage. arXiv preprint arXiv:2208.03188 (2022).

[30]

Shamane Siriwardhana, Rivindu Weerasekera, Elliott Wen, Tharindu Kalu-

arachchi, Rajib Rana, and Suranga Nanayakkara. 2023. Improving the domain

adaptation of retrieval augmented generation (RAG) models for open domain

question answering.

Transactions of the Association for Computational Linguistics

11 (2023), 1-17.

- [31] Nisan Stiennon, Long Ouyang, Jeffrey Wu, Daniel Ziegler, Ryan Lowe, Chelsea Voss, Alec Radford, Dario Amodei, and Paul F Christiano. 2020. Learning to summarize with human feedback. Advances in Neural Information Processing Systems 33 (2020), 3008-3021.
- [32] Kai Sun, Yifan Ethan Xu, Hanwen Zha, Yue Liu, and Xin Luna Dong. 2023. Headto-Tail: How Knowledgeable are Large Language Models (LLM)? A.K.A. Will LLMs Replace Knowledge Graphs? arXiv:2308.10168 [cs.CL]
- [33] Edward Tian. 2023. GPTZero. Online; accessed 23-Mar-2023. https://gptzero.me/
- [34] SM Tonmoy, SM Zaman, Vinija Jain, Anku Rani, Vipula Rawte, Aman Chadha, and Amitava Das. 2024. A comprehensive survey of hallucination mitigation techniques in large language models. arXiv preprint arXiv:2401.01313 (2024).
- [35] Xintao Wang, Qianwen Yang, Yongting Qiu, Jiaqing Liang, Qianyu He, Zhouhong Gu, Yanghua Xiao, and Wei Wang. 2023. KnowledGPT: Enhancing Large Language Models with Retrieval and Storage Access on Knowledge Bases. arXiv:2308.11761 [cs.CL]
- [36] Jason Wei, Xuezhi Wang, Dale Schuurmans, Maarten Bosma, Brian Ichter, Fei Xia, Ed Chi, Quoc Le, and Denny Zhou. 2023. Chain-of-Thought Prompting Elicits Reasoning in Large Language Models. arXiv:2201.11903 [cs.CL]
- [37] Andreas Wortmann, Olivier Barais, Benoit Combemale, and Manuel Wimmer. 2020. Modeling languages in Industry 4.0: an extended systematic mapping study. Software and Systems Modeling 19 (2020), 67-94.
- [38] Laurie Wu, Alei Fan, Yang Yang, and Zeya He. 2022. Tech-touch balance in the service encounter: The impact of supplementary human service on consumer responses. International Journal of Hospitality Management 101 (2022), 103122.
- [39] Xiaoshi Wu, Keqiang Sun, Feng Zhu, Rui Zhao, and Hongsheng Li. 2023. Human preference score: Better aligning text-to-image models with human preference. In Proceedings of the IEEE/CVF International Conference on Computer Vision . 20962105.
- [40] Zhentao Xu, Mark Jerome Cruz, Matthew Guevara, Tie Wang, Manasi Deshpande, Xiaofeng Wang, and Zheng Li. 2024. Retrieval-Augmented Generation with Knowledge Graphs for Customer Service Question Answering. arXiv preprint arXiv:2404.17723 (2024).
- [41] Fan Yang, Zheng Chen, Ziyan Jiang, Eunah Cho, Xiaojiang Huang, and Yanbin Lu. 2023. Palr: Personalization aware llms for recommendation. arXiv preprint arXiv:2305.07622 (2023).
- [42] Shunyu Yao, Jeffrey Zhao, Dian Yu, Nan Du, Izhak Shafran, Karthik Narasimhan, and Yuan Cao. 2023. ReAct: Synergizing Reasoning and Acting in Language Models. arXiv:2210.03629 [cs.CL]
- [43] Michihiro Yasunaga, Armen Aghajanyan, Weijia Shi, Rich James, Jure Leskovec, Percy Liang, Mike Lewis, Luke Zettlemoyer, and Wen tau Yih. 2023. RetrievalAugmented Multimodal Language Modeling. arXiv:2211.12561 [cs.CV]
- [44] Wenhao Yu. 2022. Retrieval-augmented Generation across Heterogeneous Knowledge. In Proceedings of the 2022 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies: Student Research Workshop , Daphne Ippolito, Liunian Harold Li, Maria Leonor Pacheco, Danqi Chen, and Nianwen Xue (Eds.). Association for Computational Linguistics, Hybrid: Seattle, Washington + Online, 52-58. https://doi.org/10.18653/v1/2022. naacl-srw.7
- [45] Zichun Yu, Chenyan Xiong, Shi Yu, and Zhiyuan Liu. 2023. AugmentationAdapted Retriever Improves Generalization of Language Models as Generic Plug-In. arXiv:2305.17331 [cs.CL]
- [46] Yuheng Zha, Yichi Yang, Ruichen Li, and Zhiting Hu. 2023. AlignScore: Evaluating Factual Consistency with A Unified Alignment Function. In Proceedings of the 61st Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers) . 11328-11348.
- [47] Kai Zhang, Fubang Zhao, Yangyang Kang, and Xiaozhong Liu. 2023. Memoryaugmented llm personalization with short-and long-term memory coordination. arXiv preprint arXiv:2309.11696 (2023).
- [48] Peitian Zhang, Shitao Xiao, Zheng Liu, Zhicheng Dou, and Jian-Yun Nie. 2023. Retrieve Anything To Augment Large Language Models. arXiv:2310.07554 [cs.IR]
- [49] Penghao Zhao, Hailin Zhang, Qinhan Yu, Zhengren Wang, Yunteng Geng, Fangcheng Fu, Ling Yang, Wentao Zhang, and Bin Cui. 2024. Retrieval-augmented generation for ai-generated content: A survey. arXiv preprint arXiv:2402.19473 (2024).

## A Evaluation of RAG Approach for Open Source Datasets

To evaluate the effectiveness of the RAG-based approach, we also conducted a sample study using three open-domain datasets following a similar methodology.

## A.1 Dataset Statistics

Weconsider three popular open source question answering datasets focusing on a varitey of topics. A random subset of MSMARCO

[3], SQuAD [23], TriviaQA [18] were considered for the evaluation. Table 7 shows the brief overview of the considered datasets.

Table 7: Open source dataset random sample statistics

| Dataset   |   Unique Docs |   Ave. Doc. Tokens |   Unique Quest- ions |   Ave. Question Tokens |   Unique Answers |   Ave. Answer Tokens |
|-----------|---------------|--------------------|----------------------|------------------------|------------------|----------------------|
| MS-MARCO  |          4997 |              58.80 |                 5000 |                   5.67 |             4999 |                14.37 |
| SQUAD     |          5000 |              94.21 |                 4994 |                  10.33 |             4413 |                 2.59 |
| TRIVIA    |          3530 |             4321.4 |                 4087 |                  12.95 |             3471 |                 1.67 |

## A.2 Retrieval and Embedding Performance

We observed a specific trend in Recall values in lower k values (1, 3) versus higher k values (5, 10) for SQuAD and TRIVIA. For SQuAD, Vertex AI textembedding-gecko@001 (768) embedding with ScaNN retrieval performed the best at lower k but at higher k, SBERT-allmpnet-base-v2 (768) with ScaNN performed better. For TRIVIA, SBERTall-mpnet-base-v2 (768) embedding with HNSW KNN retrieval performed the best at lower k but at higher k, SBERT-allmpnet-base-v2 (768) with ScaNN performed better. For MSMARCO, Vertex AI -textembedding-gecko@001 (768) embedding with ScaNN retrieval combination was a clear winner. Refer Table 8 for more details.

## A.3 Evaluation of Generated Responses

Table 9 shows the accuracy, hallucination, and missing rate for the open sources datasets (through automated evaluation as described in Subsection 4.2.1. As observed in the Retrieval evaluation section, we notice a similar relationship in the accuracy and hallucination as document size increases. From Table 7, we observe that the token length of the TriviaQA documents is much larger than MS-MARCO and SQuAD. Similarlt, we observed lower accuracy and higher hallucination rates with TriviaQA when compared to MS-MARCO and SQuAD. Table 10 and 11 show performance of Chain of Thought Prompting and Chain of Verification performance on open-source datasets. Accuracy and hallucination rate improvement vary based on the open source dataset.

## B Prompt Examples

We utilize LLMs for various tasks in our methodology, including question-answer pair generation from knowledge base articles, response generation, factual accuracy evaluation, and advanced CoTP &amp; CoV prompts. Therefore, we include the specific prompts used for these different tasks.

Prompt for answer generation. You are a reading comprehension and answer generation expert. Please answer the question from the document provided. If the document is not related to the question, simply reply: "Sorry, I cannot answer this question". Following are the guidelines you need to follow for generating the responses:

1) They should always be professional, positive, friendly, and empathetic. 2) They should not contain words that have a negative connotation (Example: "unfortunately"). 3) They should always be truthful and honest. 4) They should always be STRICTLY less than 30 words. If the generated response if greater than 30 words, rephrase and make it less than 30 words.

document: &lt;retrieved\_document&gt;,

question: &lt;question&gt;,

output:

Prompt for Hallucination Judgement. :

You need to check whether the prediction of a question-answering systems to a question is correct. You should make the judgement based on a list of ground truth answers provided to you. You response should be "correct" if the prediction is correct or "incorrect" if the prediction is wrong. Your response should be "unsure" where there is a valid ground truth and prediction is "Sorry, I don't know." or if you are not confident if the prediction is correct.

Below are the different cases possible:

1) Examples where you should return "correct".

Question: What is the customer registration process?

Ground Truth: The customer registration process is a way for customers to create an account with them. This allows them to track their purchases, receive personalized offers, and more. The process is simple and can be completed in a few minutes.

Prediction: The customer registration process is a process that allows customers to register their information with them. This process allows customers to receive benefits such as discounts, special offers, and personalized shopping experiences.

Correctness: correct

Question: What happens if my refund is pending?

Ground Truth: Sorry, I don't know.

Prediction: Sorry, I don't know.

Correctness: correct

2) Examples where you should return "incorrect".

Question: What do I need to do to get the military discount?

Ground Truth: You need to have a smartphone and be registered for the discount. If you don't have a smartphone, you can use discount code RC5. If you are in the pilot 425 stores area, you can key in your phone number.

Prediction: The military discount is available to active duty military members, veterans, and their families. The discount is 10 percent off eligible purchases.

Correctness: incorrect

Question: How do I apply for the consumer card?

Ground Truth: Sorry, I don't know.

Prediction: You can apply for the consumer card in-store, online

or by mail.

Correctness: incorrect

3) Examples where you should return "unsure".

Question: What is the Return Policy?

Ground Truth: The Return Policy is available on the website. You can find it by searching for "Return Policy" or by clicking on the link in the article.

Prediction: Sorry, I don't know.

Correctness: unsure

Provide correctness for the below question, ground truth and prediction:

Question: &lt;question&gt;

Ground Truth: &lt;ground truth&gt; Prediction: &lt;prediction&gt; Correctness:

Prompt for Chain of Prompting : Prompt for quote extraction: You are a reading comprehension and quote extraction expert. Please extract, word-for-word, any quotes relevant to the question. If there are no quotes in this document that seem relevant to the provided question, please say "I can't find any relevant quotes".

For document: &lt;document&gt;,

question: &lt;question&gt;,

output:

Prompt for Generating Baseline Response and Plan Verification (Chain of Verification). :

Below is a question: &lt;question&gt;

Below is the document from which the answer should be generated: &lt;document&gt;

You are an subject matter expert working at Contact Centers. Your expertise includes quote extraction, answer generation, and asking verification questions to improve the overall factual accuracy of the answers you provide.

| Dataset   | Embedding Strategy                        | Retrieval Strategy   |   Recall@1 |   Recall@3 |   Recall@5 |   Recall @10 |
|-----------|-------------------------------------------|----------------------|------------|------------|------------|--------------|
| SQUAD     | USE (512)                                 | HNSW KNN             |     0.4188 |     0.5946 |     0.6634 |       0.7424 |
| SQUAD     | SBERT - all-mpnet-base-v2 (768)           | HNSW KNN             |     0.6708 |     0.8444 |     0.8902 |       0.9336 |
| SQUAD     | Vertex AI - textembedding-gecko@001 (768) | HNSW KNN             |     0.6958 |     0.8486 |     0.8804 |        0.911 |
| SQUAD     | USE (512)                                 | ScaNN                |     0.4282 |     0.6116 |     0.6834 |       0.7666 |
| SQUAD     | SBERT - all-mpnet-base-v2 (768)           | ScaNN                |      0.685 |     0.8636 |      0.913 |       0.9584 |
| SQUAD     | Vertex AI - textembedding-gecko@001 (768) | ScaNN                |     0.7156 |      0.874 |      0.908 |       0.9414 |
| TRIVIA    | USE (512)                                 | HNSW KNN             |      0.459 |     0.6004 |     0.6604 |       0.7333 |
| TRIVIA    | SBERT - all-mpnet-base-v2 (768)           | HNSW KNN             |      0.793 |     0.8691 |     0.8921 |       0.9171 |
| TRIVIA    | Vertex AI - textembedding-gecko@001 (768) | HNSW KNN             |     0.6423 |     0.7487 |      0.782 |       0.8233 |
| TRIVIA    | USE (512)                                 | ScaNN                |     0.4101 |     0.6039 |     0.6687 |       0.7548 |
| TRIVIA    | SBERT - all-mpnet-base-v2 (768)           | ScaNN                |     0.7086 |     0.8654 |     0.8992 |       0.9288 |
| TRIVIA    | Vertex AI - textembedding-gecko@001 (768) | ScaNN                |     0.5936 |      0.759 |     0.8038 |       0.8537 |
| MS-MARCO  | USE (512)                                 | HNSW KNN             |     0.5263 |     0.6856 |     0.7347 |        0.784 |
| MS-MARCO  | SBERT - all-mpnet-base-v2 (768)           | HNSW KNN             |     0.9128 |     0.9798 |     0.9878 |       0.9925 |
| MS-MARCO  | Vertex AI - textembedding-gecko@001 (768) | HNSW KNN             |     0.8194 |     0.9241 |     0.9425 |       0.9577 |
| MS-MARCO  | USE (512)                                 | ScaNN                |     0.5347 |     0.6996 |     0.7518 |       0.8035 |
| MS-MARCO  | SBERT - all-mpnet-base-v2 (768)           | ScaNN                |     0.9132 |     0.9816 |     0.9896 |       0.9944 |
| MS-MARCO  | Vertex AI - textembedding-gecko@001 (768) | ScaNN                |     0.8296 |     0.9376 |     0.9581 |       0.9738 |

Table 8: Recall@K for retrieval and embedding strategies for different data sets.

|   k | Data     |   Accuracy |   Hallucination rate |   Missing rate |
|-----|----------|------------|----------------------|----------------|
|   1 | SQUAD    |      84.74 |                 8.36 |            6.9 |
|   1 | TriviaQA |      58.48 |                28.68 |           12.8 |
|   1 | MS-MARCO |      89.06 |                 7.06 |           3.86 |
|   3 | SQUAD    |      91.32 |                 5.48 |            3.2 |
|   3 | TriviaQA |      25.08 |                67.41 |           7.46 |
|   3 | MS-MARCO |       89.9 |                 6.93 |           3.16 |

Table 9: Generation quality metrics (%) using text-bison@001 and ScaNN

Table 10: Baseline vs CoTP evaluation metrics (%) without retrieval (question-doc. pair used as it is)

| dataset   | accuracy   | accuracy   | hallucination_rate   | hallucination_rate   | missing_rate   | missing_rate   |
|-----------|------------|------------|----------------------|----------------------|----------------|----------------|
|           | baseline   | CoTP       | baseline             | CoTP                 | baseline       | CoTP           |
| SQUAD     | 98.14      | 94.58      | 1                    | 4.16                 | 0.86           | 1.9            |
| TRIVIA    | 63.95      | 86.27      | 22.85                | 6.45                 | 13.18          | 6.38           |
| MS-MARCO  | 92.1       | 90.94      | 4.64                 | 4.32                 | 3.26           | 4.74           |

Table 11: Baseline vs CoVe evaluation metrics (%) without retrieval (question-doc. pair used as it is)

| dataset   | accuracy   | accuracy   | hallucination_rate   | hallucination_rate   | missing_rate   | missing_rate   |
|-----------|------------|------------|----------------------|----------------------|----------------|----------------|
|           | baseline   | CoVe       | baseline             | CoVe                 | baseline       | CoVe           |
| SQUAD     | 98.14      | 95.96      | 1                    | 3.24                 | 0.86           | 0.8            |
| TRIVIA    | 63.95      | 63.76      | 22.85                | 23.83                | 13.18          | 12.4           |
| MS-MARCO  | 92.1       | 92.6       | 4.64                 | 5.54                 | 3.26           | 1.86           |

Your first goal is to extract, word-for-word, any quotes relevant to the question that could be used to answer the question. If there are no quotes in this document that seem relevant to the provided question, simply return: "I can't find any relevant quotes".

Your second goal is to use *solely* the quotes extracted from the first goal and generate a concise and accurate answer (using the below listed guideline) by rephrasing the quotes to answer the question. If the quotes could not be used to answer the question, simply return: "Sorry, I cannot answer this question". 1) They should always be professional, positive, friendly, and empathetic. 2) They should not contain words that have a negative connotation (Example: "unfortunately"). 3) They should always be truthful and honest. 4) They should always be STRICTLY less than 30 words. If the generated response if greater than 30 words, rephrase and make it less than 30 words.

Your third goal is to generate a list of potential areas that might require verification based on the content of the document to increase factual accuracy of the answer. Your response should be in the below format:

'' Quotes: &lt;Your Extracted Quotes&gt;

Answer: &lt;Your Answer&gt;

Potential Areas for Verification: 1) Your Specific point or segment from your answer. 2) Your Another point or segment from your answer. N) Your Nth point or segment from your answer. ''

Prompt for Executing Verification Questions and Generating Verified Response (Chain of Verification). :

Below is a question: &lt;question&gt;

Below is the answer: &lt;answer&gt;

Below is the document from which the answer was generated: &lt;document&gt;

Based on the potential areas for verification: &lt;areas of verification&gt;

You are an subject matter expert working at Contact Centers. Your expertise includes improvising answers to questions about the company to increase factual correctness using the factual accuracy verification questions provided to you.

Your goal is to check each verification point against the document, provide feedback on any inconsistencies, and then generate a final verified (using the below listed guidelines), concise and accurate answer in strictly less than 30 words that addresses the factual inconsistencies. 1) They should always be professional, positive, friendly, and empathetic. 2) They should not contain words that have a negative connotation (Example: "unfortunately"). 3) They should always be truthful and honest. 4) They should always be STRICTLY less than 30 words. If the generated response if greater than 30 words, rephrase and make it less than 30 words.

Your response should be in the below format:

'' Feedback: 1) Your Verification for point 1. 2) Your Verification for point 2. N) Your Verification for point N.

Final Verified Response: [Your Revised Response] ''

Prompt for generating answer from a document and question (open source datasets). You are a question answering bot. Your job is to generate answer to the question using the provided articles. The answers should be derived only from the articles. If the answer is not present in the articles, return the text - NOANSWERFOUND.

The answer should be less than 10 words and in a sentence format.

Example where answer could not be found in the articles:

Question: Which county is Smyrna city in?

Document: Georgia is a southeastern U.S. state whose terrain spans coastal beaches, farmland and mountains. Capital city Atlanta is home of the Georgia Aquarium and the Martin Luther King Jr. National Historic Site, dedicated to the African-American leader's life and times.

Return Text: NOANSWERFOUND

Example where answer could be found in the articles:

Question: Which county is Smyrna city in?

Document: Smyrna is a city in Cobb County, Georgia, United States. Cobb County is a county in the U.S. state of Georgia, located in the Atlanta metropolitan area in the north central portion of the state.

Return Text: Cobb County of the state of Georgia

Provide answer to the below Question/Query using the below Document.

Question: &lt;question&gt;

Document: &lt;document&gt;

Return Text:

# Chunking (문서 분할)

![rag_split](figures/rag_split.png)

- Load 한 문서를 지정한 기준의 덩어리(chunk)로 나누는 작업을 진행한다.

## 나누는 이유
1. **임베딩 모델의 컨텍스트 길이 제한**
    - 대부분의 언어 모델은 한 번에 처리할 수 있는 토큰 수에 제한이 있다. 전체 문서를 통째로 입력하면 이 제한을 초과할 수 있어 처리가 불가능해진다.
2. **검색 정확도 향상**
    - 큰 문서 전체보다는 특정 주제나 내용을 다루는 작은 chunk가 사용자 질문과 더 정확하게 매칭된다. 예를 들어, 100페이지 매뉴얼에서 특정 기능에 대한 질문이 있을 때, 해당 기능을 설명하는 몇 개의 문단만 검색되는 것이 더 효과적이다.
    - 사용자 질문에 대해 문서의 모든 내용이 다 관련있는 것은 아니다. Chunking을 통해 가장 관련성 높은 부분만 선별적으로 활용할 수 있어 답변의 품질이 향상된다.
    - 전체 문서에는 질문과 무관한 내용들이 많이 포함되어 있어 모델이 혼란을 겪을 수 있다. 적절한 크기의 chunk는 이런 노이즈를 줄여준다.
3. **계산 효율성**
    - 벡터 유사도 계산, 임베딩 생성 등의 작업이 작은 chunk 단위로 수행될 때 더 빠르고 효율적이다. 메모리 사용량도 줄일 수 있다.

## 주요 Splitter
- **Splitter**는 문서를 분할(chunking)을 처리해주는 도구들이다. Langchain은 분할 대상, 방법에 따라 다양한 splitter를 제공한다.
- **Splitter 의 목표**
  - 가능한 한 **의미 있는 덩어리를 유지**하면서, **최대 길이(chunk_size)**를 넘지 않도록 나누기.
- https://reference.langchain.com/python/langchain_text_splitters/

### CharacterTextSplitter
가장  기본적인 Text spliter
- 한개의 구분자를 기준으로 분리한다. (default: "\n\n")
    - 분리된 조각이 chunk size 보다 작으면 다음 조각과 합칠 수 있다.
        - 합쳤을때 chuck_size 보다 크면 안 합친다. chuck_size 이내면 합친다.
    - 나누는 기준은 구분자이기 때문에 chunk_size 보다 글자수가 많을 수 있다.
- chunk size: 분리된 문서(chunk) 글자수 이내에서 분리되도록 한다.
    -  구분자를 기준으로 분리한다. 구분자를 기준으로 분리한 문서 조각이 chunk size 보다 크더라도 그대로 유지한다. 즉 chunk_size가 우선이 아니라 **seperator** 가 우선이다.
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - seperator: 구분 문자열을 지정. (default: '\n\n')
- CharacterTextSplitter는 단순 스플리터로 overlap기능을 지원하지는 않는다. 단 seperator가 빈문자열("") 일 경우에는 overlap 기능을 지원한다. overlap이란 각 이전 청크의 뒷부분의 문자열을 앞에 붙여 문맥을 유지하는 것을 말한다.
  
### RecursiveCharacterTextSplitter
- RecursiveCharacterTextSplitter는 **긴 텍스트를 지정된 최대 길이(chunk_size) 이하로 나누는 데 효과적인 텍스트 분할기**(splitter)이다.
- 여러 **구분자(separators)를 순차적으로 적용**하여, 가능한 한 자연스러운 문단/문장/단어 단위로 분할하고, 최종적으로는 크기 제한을 만족시킨다.
- 분할 기준 문자
    1. 두 개의 줄바꿈 문자 ("\n\n")
    2. 한 개의 줄바꿈 문자 ("\n")
    3. 공백 문자 (" ")
    4. 빈 문자열 ("")
- 작동 방식
    1. 먼저 가장 높은 우선순위의 구분자("\n\n")를 기준으로 분리한다.
    2. 분할된 조각 중 **chunk_size를 초과하는 조각**에 대해 다음 우선순위 구분자("\n" → " " → "")로 재귀적으로 재분할한다.
    3. 이 과정을 통해 모든 조각(chunk)이 chunk_size를 초과하지 않도록 만든다.  
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - chunk_overlap: 연속된 청크들 간의 겹치는 문자 수를 설정. 새로운 청크 생성 시 이전 청크의 마지막 부분에서 지정된 수만큼의 문자를 가져와서 새 청크의 앞부분에 포함시켜, 청크 경계에서 문맥의 연속성을 유지한다.
      - 구분자에 의해 청크가 나눠지면 정상적인 분리이므로 overlap이 적용되지 않는다.
      - 정상적 구분자로 나눌 수 없어 chunk_size에 맞춰 잘라진 경우 문맥의 연결성을 위애 overlap을 적용한다.
    - separators(list): 구분자를 지정한다. 지정하면 기본 구분자가 지정한 것으로 변경된다.

#### 메소드
- `split_documents(Iterable[Document]) : List[Document]`
    - Document 목록을 받아 split 처리한다.
- `split_text(str) : List[str]`
    - string text를 받아서 split 처리한다. 

In [14]:
!uv pip install langchain-text-splitters

Checked 1 package in 34ms


In [15]:
text = """123456789012345678901234567890123456789012345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

splitter = CharacterTextSplitter(
    chunk_size=60,
    chunk_overlap=10,  # default: 200, chunk_overlap이 chunk_size보다 크면 안됨.
    # chunk_overlap은 separator가 빈문자열일때 적용된다.
    # separator="" # 기본: `\n\n` -> "" 변경. "": chunk_size에 맞추겠다.
)

result = splitter.split_text(text) # 입력: str
print(type(result))
print(type(result[0]))
print(len(result))

<class 'list'>
<class 'str'>
4


In [20]:
for chunk in result:
    print(len(chunk), chunk)
    print("-------------------------------------------------")

60 123456789012345678901234567890123456789012345678901234567890
-------------------------------------------------
60 1234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLM
-------------------------------------------------
60 DEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefg
-------------------------------------------------
55 이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
-------------------------------------------------


In [ ]:
doc = Document(page_content=text)
# 나눈 문서가 Document 객체일때
result_docs = splitter.split_documents([doc])

In [24]:
print(len(result_docs))
print(type(result_docs[0]))

4
<class 'langchain_core.documents.base.Document'>


In [26]:
for doc in result_docs:
    print(doc)
    print("--------------")

page_content='123456789012345678901234567890123456789012345678901234567890'
--------------
page_content='1234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLM'
--------------
page_content='DEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefg'
--------------
page_content='이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
--------------


In [33]:
len("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")

52

In [34]:
text2 = """1234567890123456789012345678901234567890
12345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRST.UVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ RSTUVWXYZ
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''] # 구분자 리스트를 직접 지정
    , is_separator_regex=True  # 구분자에 정규표현식 사용가능 여부.
)
result2 = splitter.split_text(text2)

In [36]:
print(splitter._separators)

['\n\n', '\n', '[\\.?!,~]', ' ', '']


In [37]:
for txt in result2:
    print(len(txt), txt)
    print('--------------------------------------------')

40 1234567890123456789012345678901234567890
--------------------------------------------
29 12345678901234567890123456789
--------------------------------------------
46 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRST
--------------------------------------------
7 .UVWXYZ
--------------------------------------------
26 가나다라마바사아자차카타파하

아야어여오요우유으이
--------------------------------------------
43 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ
--------------------------------------------
9 RSTUVWXYZ
--------------------------------------------
49 abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVW
--------------------------------------------
13 NOPQRSTUVWXYZ
--------------------------------------------


In [46]:
## 문서 Load => Split
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

path = "data/olympic.txt"

# load
loader = TextLoader(path, encoding="UTF-8")
docs = loader.load()
print("Load한 문서 개수", len(docs))

# split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", r"[\.?!,~]", ' ', ''], 
    is_separator_regex=True
)
split_docs = splitter.split_documents(docs)
print("Split후 문서 개수", len(split_docs))

Load한 문서 개수 1
Split후 문서 개수 61


In [45]:
split_docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

In [47]:
# 문서 Load와 Split을 한번에 처리
docs = loader.load_and_split(splitter)
print(len(docs))

61


In [49]:
docs[1].page_content

'올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다'

## Token 수 기준으로 나누기

- LLM 언어 모델들은 입력 토큰 수 제한이 있어서 요청시 제한 토큰수 이상의 프롬프트는 전송할 수 없다.
- 따라서 텍스트를 chunk로 분할할 때는 글자수 보다 **토큰 수를 기준으로 크기를 지정하는 것**이 좋다.  
- 토큰기반 분할은 텍스트의 의미를 유지하면서 분할하는 방식이므로 문자 기반 분할과 같이 단어가 중간잘리는 것들을 방지할 수 있다. 
- 토큰 수 계산할 때는 사용하는 언어 모델에 사용된 것과 동일한 tokenizer를 사용하는 것이 좋다.
  - 예를 들어 OpenAI의 GPT 모델을 사용할 경우 tiktoken 라이브러리를 활용하여 토큰 수를 정확하게 계산할 수 있다.

### [tiktoken](https://github.com/openai/tiktoken) tokenizer 기반 분할
- OpenAI에서 GPT 모델을 학습할 때 사용한 `BPE` 방식의 tokenizer. **OpenAI 언어모델을 사용할 경우 이것을 사용하는 것이 좀 더 정확하게  토큰을 계산할 수 있다.**
- Splitter.from_tiktoken_encoder() 메소드를 이용해 생성
  - `RecursiveCharacterTextSplitter.from_tiktoken_encoder()`
  - `CharacterTextSplitter.from_tiktoken_encoder()`
- 파라미터
  - encode_name: 인코딩 방식(토큰화 규칙)을 지정. OpenAI는 GPT 모델들 마다 다른 방식을 사용했다. 그래서 사용하려는 모델에 맞는 인코딩 방식을 지정해야 한다.
    - `o200k_base`: GPT-4 이후 모델들이 사용한 방식
    - `cl100k_base`: 초기 GPT-4 및 GPT-3.5-Turbo 모델에서 사용된 방식.
    - `r50k_base:` GPT-3 모델에서 사용된 방식 
  - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- tiktoken 설치
  - `pip install tiktoken`

### HuggingFace Tokenizer
- HuggingFace 모델을 사용할 경우 그 모델이 사용한 tokenizer를 이용해 토큰 기반으로 분할 한다.
  - 다른 tokenizer를 이용해 분할 할 경우 토큰 수 계산이 다르게 될 수있다.
- `from_huggingface_tokenizer()` 메소드를 이용.
  - 파라미터
    - tokenizer: HuggingFace tokenizer 객체
    - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- `transformers` 라이브러리를 설치해야 한다.
  - `pip install transformers` 

In [50]:
!uv pip install tiktoken

Checked 1 package in 29ms


In [56]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

loader = TextLoader(path, encoding="utf-8")
#from_tiktoken_encoder() -> OpenAI GPT모델을 사용할 경우 사용.
#                           GPT모델명 또는 Encoder(토큰나이저)의 이름
# splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     # model_name="gpt-5",
#     encoding_name="o200k_base",
#     chunk_size=500, # 500토큰기준
#     chunk_overlap=50, # 50토큰
# )
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    # model_name="gpt-5",
    encoding_name="o200k_base",
    chunk_size=500, # 500토큰기준
    chunk_overlap=50, # 50토큰
)
docs = loader.load_and_split(splitter)
print(len(docs))

Created a chunk of size 1027, which is longer than the specified 500
Created a chunk of size 981, which is longer than the specified 500
Created a chunk of size 881, which is longer than the specified 500
Created a chunk of size 608, which is longer than the specified 500
Created a chunk of size 703, which is longer than the specified 500
Created a chunk of size 843, which is longer than the specified 500
Created a chunk of size 925, which is longer than the specified 500
Created a chunk of size 661, which is longer than the specified 500
Created a chunk of size 1305, which is longer than the specified 500
Created a chunk of size 1052, which is longer than the specified 500


18


In [58]:
print(len(docs[0].page_content))

1726


In [60]:
# huggingface tokenzier 이용
from transformers import AutoTokenizer
model_id = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)# model_id는 사용할 LLM 모델의 이름 
splitter_hf = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer, # transformers.Tokenizer객체
    chunk_size=500,
    chunk_overlap=50
)

In [61]:
docs = loader.load_and_split(splitter_hf)
len(docs)

35

## MarkdownHeaderTextSplitter
- Markdown Header 기준으로 Splitter
- Loading한 문서가 Markdown 문서이고 Header를 기준으로 문서의 내용이 나눠질때 사용.
- https://reference.langchain.com/python/langchain_text_splitters/#langchain_text_splitters.MarkdownTextSplitter

In [ ]:
text = """
# 대주제1
- 동물

## 중주제1
- 포유류

- 조류

### 소주제1
- 개
- 고양이
- 까치
- 독수리

# 대주제2

## 중주제2
- 기차
- 배
"""

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 나눌때 기준이 되는 Header 를 설정. 
# list(tuple(key, value)): key-Header기호, value: 이름 
header_to_split = [
    ("#", "header1"),
    ("##", "header2"),
    # ("###", "header3"),
    # ("####", "header4"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=header_to_split,
    # strip_headers=False, #default: True - 구분자 Header(제목)을 내용에 표시할 지 여부. True: 안넣는다.
)

docs = splitter.split_text(text)
len(docs)

4

In [76]:
docs

[Document(metadata={'header1': '대주제1'}, page_content='# 대주제1\n- 동물'),
 Document(metadata={'header1': '대주제1', 'header2': '중주제1'}, page_content='## 중주제1\n- 포유류  \n- 조류  \n### 소주제1\n- 개\n- 고양이\n- 까치\n- 독수리'),
 Document(metadata={'header1': '대주제2'}, page_content='# 대주제2\nalksdjflaksdjflakdsjlkasdfjasdfkl'),
 Document(metadata={'header1': '대주제2', 'header2': '중주제2'}, page_content='## 중주제2\n- 기차\n- 배')]

In [77]:
# olympic.md

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

path = "data/olympic_wiki.md"
loader =TextLoader(path, encoding="utf-8")
header_to_split = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=header_to_split
)

In [82]:
"-".join(["a", "b", "c"])

'a-b-c'

In [85]:
docs = loader.load()

# list[Document] -> Document에서 page_content(읽은 text)를 
#                   조회해서 하나의 str로 반환
doc_txt = "\n".join(doc.page_content for doc in docs)

# MarkdownHeaderSplitter는  split_document가 없고 split_text(str)만 제공
split_docs = splitter.split_text(doc_txt)
len(split_docs)

25

In [88]:
split_docs[0].metadata

{'h1': '올림픽'}

In [89]:
print(split_docs[0].page_content)

올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  
또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경제, 정치,